## yfinance.ipynb 

yfinance.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1-jwZq-8y7BLGSEf5XnRaHhe9P25Up42s


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "yfinance", "pandas", "numpy", "tqdm", "pyarrow",
    "lxml", "html5lib", "requests", "-q"])
import yfinance as yf
import pandas as pd
import numpy as np
import os, time, json, warnings, requests
from tqdm import tqdm
import dotenv
dotenv.load_dotenv()  # for any future use of environment variables (e.g. API keys)
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
!pip install yfinance -q
!pip install yfinance pandas numpy tqdm pyarrow -q

In [ ]:
BASE        = str(os.getenv("yFinDataPath","data/yFinance" ))
RAW         = f"{BASE}/raw"
META        = f"{BASE}/raw_metadata"
PROCESSED   = f"{BASE}/processed"
QUALITY     = f"{BASE}/quality"

for p in [RAW, META, PROCESSED, QUALITY]:
    os.makedirs(p, exist_ok=True)

Build ticker universe (Wikipedia 403 fix)

In [ ]:
import requests, io
# Spoof a real browser so Wikipedia doesn't block us
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def fetch_wiki_tickers(url, possible_cols=("Symbol","Ticker","Ticker symbol")) -> list:
    try:
        html  = requests.get(url, headers=HEADERS, timeout=15).text
        dfs   = pd.read_html(io.StringIO(html))
        df    = dfs[0]
        for col in possible_cols:
            if col in df.columns:
                tickers = df[col].dropna().tolist()
                tickers = [str(t).replace(".","-").strip() for t in tickers]
                return [t for t in tickers if 1 <= len(t) <= 7]
        # auto-detect: find column whose values look like tickers
        for col in df.columns:
            sample = df[col].dropna().head(5).tolist()
            if all(str(v).isupper() and 1 <= len(str(v)) <= 6 for v in sample):
                tickers = df[col].dropna().tolist()
                return [str(t).replace(".","-").strip() for t in tickers]
        print(f"  ⚠️  No ticker col found. Columns: {list(df.columns)}")
        return []
    except Exception as e:
        print(f"  ⚠️  Failed: {url} — {e}")
        return []

print("Fetching ticker lists from Wikipedia…")
sp500    = fetch_wiki_tickers("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
print(f"  S&P 500   : {len(sp500)}")

sp400    = fetch_wiki_tickers("https://en.wikipedia.org/wiki/List_of_S%26P_400_companies")
print(f"  S&P 400   : {len(sp400)}")

sp600    = fetch_wiki_tickers("https://en.wikipedia.org/wiki/List_of_S%26P_600_companies")
print(f"  S&P 600   : {len(sp600)}")

nasdaq100 = fetch_wiki_tickers(
    "https://en.wikipedia.org/wiki/Nasdaq-100",
    possible_cols=("Ticker","Symbol","Ticker symbol")
)
print(f"  NASDAQ-100: {len(nasdaq100)}")

djia30   = fetch_wiki_tickers(
    "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average",
    possible_cols=("Symbol","Ticker")
)
print(f"  DJIA 30   : {len(djia30)}")

# GitHub CSVs as backup for Russell 1000 + extra coverage
GITHUB_CSVS = [
    # S&P 500 maintained CSV
    "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv",
    # Additional Russell 1000 proxy
    "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.txt",
    "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.txt",
    "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/amex/amex_tickers.txt",
]

extra = []
for csv_url in GITHUB_CSVS:
    try:
        r = requests.get(csv_url, timeout=15)
        r.raise_for_status()
        if csv_url.endswith(".txt"):
            # plain text: one ticker per line
            tickers = [line.strip() for line in r.text.splitlines() if line.strip()]
        else:
            df = pd.read_csv(io.StringIO(r.text))
            col = next((c for c in ["Symbol","Ticker","symbol","ticker"] if c in df.columns), None)
            tickers = df[col].dropna().tolist() if col else []
        tickers = [str(t).replace(".","-").strip() for t in tickers if 1 <= len(str(t)) <= 7]
        extra.extend(tickers)
        print(f"  GitHub ({csv_url.split('/')[-1]}): {len(tickers)}")
    except Exception as e:
        print(f"     GitHub failed: {e}")

# ETF universe (same as before)
ETFS = [
    "SPY","QQQ","DIA","IWM","IWB","MDY","IWR","IWS",
    "VTI","VB","VO","VV","ITOT","SCHB","SCHA","SCHM",
    "XLF","XLK","XLE","XLV","XLI","XLP","XLY","XLU","XLB","XLRE","XLC",
    "IYF","IYW","IYE","IYH","IYJ","IYC","IDU","IYM","IYR",
    "BND","AGG","TLT","IEF","SHY","LQD","HYG","JNK","MBB","TIP","BNDX",
    "VEA","VWO","EFA","EEM","IEFA","IEMG","VEU","ACWI","ACWX",
    "EWJ","EWG","EWU","EWC","EWA","EWZ","EWY","MCHI","INDA",
    "GLD","IAU","SLV","USO","UNG","DBC","PDBC","CORN","WEAT","SOYB",
    "VNQ","IYR","SCHH","REM","MORT",
    "VTV","VUG","IWD","IWF","MTUM","USMV","VLUE","QUAL","SIZE",
    "SPHB","SPHQ","SPLV","SPYD","NOBL","RDIV",
    "TQQQ","SQQQ","UPRO","SPXU","SSO","SDS","TNA","TZA",
    "VXX","UVXY","SVXY",
    "ARKK","ARKG","ARKW","ARKF","ARKQ",
    "ICLN","TAN","FAN","QCLN","PBW",
    "IBB","XBI","PJP",
    "SOXX","SMH","XSD",
    "JETS","XAR","ITA",
    "XRT","RETL",
]

ALL_TICKERS = sorted(set(sp500 + sp400 + sp600 + nasdaq100 + djia30 + extra + ETFS))
print(f"\n  Total unique tickers: {len(ALL_TICKERS)}")

Download raw OHLCV (resumable, parallel)

In [ ]:
START_DATE  = "2000-01-01"
END_DATE    = "2024-12-31"
MAX_WORKERS = 5       # increase to 8 on a fast connection
SLEEP       = 0.1

COL_RENAMES = {
    "Open":"open","High":"high","Low":"low",
    "Close":"close","Adj Close":"adj_close",
    "Volume":"volume","Dividends":"dividends","Stock Splits":"stock_splits",
}

def download_ticker(ticker: str) -> dict:
    raw_file  = f"{RAW}/{ticker}.parquet"
    meta_file = f"{META}/{ticker}.json"
    if os.path.exists(raw_file) and os.path.exists(meta_file):
        return {"ticker": ticker, "status": "skipped"}
    try:
        t    = yf.Ticker(ticker)
        hist = t.history(start=START_DATE, end=END_DATE,
                         actions=True, auto_adjust=False)
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.get_level_values(0)
        if hist.empty:
            return {"ticker": ticker, "status": "no_data"}
        hist.rename(columns=COL_RENAMES, inplace=True)
        hist.index = pd.to_datetime(hist.index).tz_localize(None)
        hist.to_parquet(raw_file)
        try:    info = t.info
        except: info = {}
        json.dump(info, open(meta_file,"w"), indent=2, default=str)
        return {"ticker": ticker, "status": "success"}
    except Exception as e:
        return {"ticker": ticker, "status": "error", "error": str(e)}

print(f"Downloading {len(ALL_TICKERS)} tickers  ({START_DATE} → {END_DATE})")
print("Already-downloaded files are skipped automatically.\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(download_ticker, t): t for t in ALL_TICKERS}
    for f in tqdm(as_completed(futures), total=len(ALL_TICKERS), desc="Downloading"):
        results.append(f.result())
        time.sleep(SLEEP)

sc = pd.Series([r["status"] for r in results]).value_counts()
print("\n  Summary:", dict(sc))
errors = [r for r in results if r["status"]=="error"]
if errors:
    print("   First 10 errors:")
    for e in errors[:10]: print(f"   {e['ticker']}: {e.get('error','')}")

Download (rate-limit safe, with retry + backoff)

In [ ]:
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)  # silence the delisted spam

START_DATE  = "2000-01-01"
END_DATE    = "2024-12-31"
MAX_WORKERS = 2       # reduced from 5 — Yahoo blocks aggressive parallel requests
MAX_RETRIES = 3       # retry each ticker up to 3 times before giving up

COL_RENAMES = {
    "Open":"open","High":"high","Low":"low",
    "Close":"close","Adj Close":"adj_close",
    "Volume":"volume","Dividends":"dividends","Stock Splits":"stock_splits",
}

def download_ticker(ticker: str) -> dict:
    raw_file  = f"{RAW}/{ticker}.parquet"
    meta_file = f"{META}/{ticker}.json"

    # Already downloaded — skip
    if os.path.exists(raw_file) and os.path.exists(meta_file):
        return {"ticker": ticker, "status": "skipped"}

    # Filter out obvious junk tickers before even trying
    # Warrants (W), Units (U), Rights (R) are usually not useful
    if len(ticker) > 5 and ticker[-1] in ("W", "U", "R"):
        return {"ticker": ticker, "status": "skipped_junk"}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            t    = yf.Ticker(ticker)
            hist = t.history(
                start=START_DATE, end=END_DATE,
                actions=True, auto_adjust=False
            )
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.get_level_values(0)
            if hist.empty:
                return {"ticker": ticker, "status": "no_data"}

            hist.rename(columns=COL_RENAMES, inplace=True)
            hist.index = pd.to_datetime(hist.index).tz_localize(None)
            hist.to_parquet(raw_file)

            try:    info = t.info
            except: info = {}
            json.dump(info, open(meta_file,"w"), indent=2, default=str)

            return {"ticker": ticker, "status": "success"}

        except Exception as e:
            err = str(e)

            # Rate limited — wait longer and retry
            if "Too Many Requests" in err or "Rate limit" in err or "429" in err:
                wait = 30 * attempt   # 30s, 60s, 90s
                print(f"\n⏳ Rate limited on {ticker}. Waiting {wait}s (attempt {attempt}/{MAX_RETRIES})…")
                time.sleep(wait)
                continue

            # Invalid crumb (Yahoo auth issue) — short wait and retry
            elif "Invalid Crumb" in err or "401" in err:
                time.sleep(10)
                continue

            # Delisted / no data — not worth retrying
            elif "possibly delisted" in err or "Data doesn't exist" in err:
                return {"ticker": ticker, "status": "no_data"}

            # Any other error — give up
            else:
                return {"ticker": ticker, "status": "error", "error": err}

    return {"ticker": ticker, "status": "error", "error": f"Failed after {MAX_RETRIES} retries"}


# Filter to only tickers not yet downloaded (true resume)
pending = [
    t for t in ALL_TICKERS
    if not (os.path.exists(f"{RAW}/{t}.parquet") and os.path.exists(f"{META}/{t}.json"))
]
print(f"Already downloaded : {len(ALL_TICKERS) - len(pending)}")
print(f"Remaining to fetch : {len(pending)}")
print(f"Workers: {MAX_WORKERS} | Retries: {MAX_RETRIES} | Starting…\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(download_ticker, t): t for t in pending}
    for f in tqdm(as_completed(futures), total=len(pending), desc="Downloading"):
        res = f.result()
        results.append(res)
        # Adaptive delay — slow down after any rate limit hit
        if res["status"] == "error" and "rate" in res.get("error","").lower():
            time.sleep(5)
        else:
            time.sleep(0.5)   # base delay between every request (was 0.1 — too fast)

sc = pd.Series([r["status"] for r in results]).value_counts()
print("\n  Summary:", dict(sc))
errors = [r for r in results if r["status"] == "error"]
if errors:
    print(f"⚠️  {len(errors)} errors. First 10:")
    for e in errors[:10]:
        print(f"   {e['ticker']}: {e.get('error','')}")

Combine into long-format OHLCV panel

In [ ]:
print("Building OHLCV panel…")
frames = []
for ticker in tqdm(ALL_TICKERS, desc="Loading"):
    fp = f"{RAW}/{ticker}.parquet"
    if not os.path.exists(fp): continue
    df = pd.read_parquet(fp)
    if df.empty: continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.rename(columns=COL_RENAMES, inplace=True)
    if "adj_close" not in df.columns and "close" in df.columns:
        df["adj_close"] = df["close"]
    df["ticker"] = ticker
    df.reset_index(inplace=True)
    df.rename(columns={"Date":"date","Datetime":"date","index":"date"},
              errors="ignore", inplace=True)
    frames.append(df)

panel = pd.concat(frames, ignore_index=True)
panel["date"] = pd.to_datetime(panel["date"]).dt.tz_localize(None)
panel.sort_values(["ticker","date"], inplace=True)
panel.reset_index(drop=True, inplace=True)
for c in ["dividends","stock_splits"]:
    if c in panel.columns: panel[c] = panel[c].fillna(0)

panel.to_parquet(f"{PROCESSED}/ohlcv_panel.parquet", index=False)
print(f"  ohlcv_panel.parquet — {panel.shape}")
print(f"   Tickers: {panel['ticker'].nunique()} | "
      f"Dates: {panel['date'].min().date()} → {panel['date'].max().date()}")

# Re-run this before Cell 7
import pandas as pd, numpy as np
from tqdm import tqdm

panel = pd.read_parquet("/content/drive/MyDrive/market_data/processed/ohlcv.parquet")
print(f"Panel loaded: {panel.shape}")

# Run this BEFORE Cell 7
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "fastparquet", "-q"])
import os
print("✅ Ready")

Feature Engineering (memory-safe, ticker-by-ticker)

In [ ]:
import gc
print("Engineering features (memory-safe mode)…")

def rsi(ret, n=14):
    g = ret.clip(lower=0)
    l = (-ret).clip(lower=0)
    ag = g.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    al = l.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    return 100 - 100/(1 + ag/al.replace(0,np.nan))

FEATURES_OUT = "/content/drive/MyDrive/market_data/processed/features.parquet"
COLS = ["date","ticker","return","log_return",
        "vol_5d","vol_21d","vol_63d","vol_252d",
        "mom_5d","mom_21d","mom_63d","mom_126d","mom_252d",
        "drawdown","max_dd_63d","max_dd_252d",
        "atr","atr_pct","rsi","macd","macd_signal","macd_hist",
        "bb_upper","bb_lower","bb_pct","bb_width",
        "dollar_volume","volume_zscore","volume_ratio",
        "high_52w","low_52w","pct_from_52w_high","pct_from_52w_low"]

all_tickers = panel["ticker"].unique()
CHUNK_SIZE  = 200   # process 200 tickers at a time, then write to disk

print(f"Total tickers: {len(all_tickers)}")
print(f"Processing in chunks of {CHUNK_SIZE}…\n")

# Delete output file if starting fresh
if os.path.exists(FEATURES_OUT):
    os.remove(FEATURES_OUT)
    print("Removed old features file.")

total_rows = 0
chunks = [all_tickers[i:i+CHUNK_SIZE] for i in range(0, len(all_tickers), CHUNK_SIZE)]

for chunk_idx, ticker_chunk in enumerate(tqdm(chunks, desc="Chunks")):

    feat_list = []

    # Only load rows for this chunk of tickers
    chunk_panel = panel[panel["ticker"].isin(ticker_chunk)].copy()

    for ticker, g in chunk_panel.groupby("ticker"):
        g = g.sort_values("date").copy()

        g["return"]     = g["adj_close"].pct_change()
        g["log_return"] = np.log(g["adj_close"]/g["adj_close"].shift(1))

        for w in [5,21,63,252]:
            g[f"vol_{w}d"] = g["return"].rolling(w).std() * np.sqrt(252)
            g[f"mom_{w}d"] = g["adj_close"].pct_change(w)
        g["mom_126d"] = g["adj_close"].pct_change(126)

        g["cummax"]      = g["adj_close"].cummax()
        g["drawdown"]    = (g["adj_close"] - g["cummax"]) / g["cummax"]
        g["max_dd_63d"]  = g["drawdown"].rolling(63).min()
        g["max_dd_252d"] = g["drawdown"].rolling(252).min()

        pc = g["close"].shift(1)
        tr = pd.concat([g["high"]-g["low"],
                        (g["high"]-pc).abs(),
                        (g["low"]-pc).abs()], axis=1).max(axis=1)
        g["atr"]     = tr.rolling(14).mean()
        g["atr_pct"] = g["atr"] / g["adj_close"]

        g["rsi"] = rsi(g["return"])

        e12 = g["adj_close"].ewm(span=12,adjust=False).mean()
        e26 = g["adj_close"].ewm(span=26,adjust=False).mean()
        g["macd"]        = e12-e26
        g["macd_signal"] = g["macd"].ewm(span=9,adjust=False).mean()
        g["macd_hist"]   = g["macd"] - g["macd_signal"]

        bm = g["adj_close"].rolling(20).mean()
        bs = g["adj_close"].rolling(20).std()
        g["bb_upper"] = bm+2*bs
        g["bb_lower"] = bm-2*bs
        g["bb_pct"]   = (g["adj_close"]-g["bb_lower"])/((g["bb_upper"]-g["bb_lower"]).replace(0,np.nan))
        g["bb_width"] = (g["bb_upper"]-g["bb_lower"])/bm

        g["dollar_volume"] = g["adj_close"]*g["volume"]
        vm = g["volume"].rolling(21).mean()
        vs = g["volume"].rolling(21).std().replace(0,np.nan)
        g["volume_zscore"] = (g["volume"]-vm)/vs
        g["volume_ratio"]  = g["volume"]/vm

        g["high_52w"]          = g["adj_close"].rolling(252).max()
        g["low_52w"]           = g["adj_close"].rolling(252).min()
        g["pct_from_52w_high"] = (g["adj_close"]-g["high_52w"])/g["high_52w"]
        g["pct_from_52w_low"]  = (g["adj_close"]-g["low_52w"]) /g["low_52w"]

        feat_list.append(g[[c for c in COLS if c in g.columns]])

    # Combine this chunk and append to parquet file on Drive
    chunk_df = pd.concat(feat_list, ignore_index=True)
    total_rows += len(chunk_df)

    if chunk_idx == 0:
        # First chunk — write fresh
        chunk_df.to_parquet(FEATURES_OUT, index=False)
    else:
        # Subsequent chunks — append by reading + writing
        # (pure parquet append isn't supported, so we use fastparquet)
        try:
            import fastparquet
            fastparquet.write(FEATURES_OUT, chunk_df, append=True)
        except ImportError:
            # fallback: accumulate in a temp folder instead
            chunk_df.to_parquet(
                f"/content/drive/MyDrive/market_data/processed/feat_chunk_{chunk_idx:04d}.parquet",
                index=False
            )

    # Free RAM
    del chunk_df, feat_list, chunk_panel
    gc.collect()

    if chunk_idx % 5 == 0:
        print(f"  Chunk {chunk_idx+1}/{len(chunks)} done — {total_rows:,} rows so far")

print(f"\n✅ Feature engineering complete — {total_rows:,} total rows")

Feature engineering (no lookahead)

In [ ]:
print("Engineering features…")

def rsi(ret, n=14):
    g = ret.clip(lower=0)
    l = (-ret).clip(lower=0)
    ag = g.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    al = l.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    return 100 - 100/(1 + ag/al.replace(0,np.nan))

feat_list = []
for ticker, g in tqdm(panel.groupby("ticker"), desc="Features"):
    g = g.sort_values("date").copy()
    g["return"]     = g["adj_close"].pct_change()
    g["log_return"] = np.log(g["adj_close"]/g["adj_close"].shift(1))

    for w in [5,21,63,252]:
        g[f"vol_{w}d"] = g["return"].rolling(w).std() * np.sqrt(252)
        g[f"mom_{w}d"] = g["adj_close"].pct_change(w)

    g["mom_126d"] = g["adj_close"].pct_change(126)

    g["cummax"]      = g["adj_close"].cummax()
    g["drawdown"]    = (g["adj_close"] - g["cummax"]) / g["cummax"]
    g["max_dd_63d"]  = g["drawdown"].rolling(63).min()
    g["max_dd_252d"] = g["drawdown"].rolling(252).min()

    pc = g["close"].shift(1)
    tr = pd.concat([g["high"]-g["low"],
                    (g["high"]-pc).abs(),
                    (g["low"] -pc).abs()], axis=1).max(axis=1)
    g["atr"]     = tr.rolling(14).mean()
    g["atr_pct"] = g["atr"] / g["adj_close"]

    g["rsi"] = rsi(g["return"])

    e12=g["adj_close"].ewm(span=12,adjust=False).mean()
    e26=g["adj_close"].ewm(span=26,adjust=False).mean()
    g["macd"]        = e12-e26
    g["macd_signal"] = g["macd"].ewm(span=9,adjust=False).mean()
    g["macd_hist"]   = g["macd"] - g["macd_signal"]

    bm  = g["adj_close"].rolling(20).mean()
    bs  = g["adj_close"].rolling(20).std()
    g["bb_upper"] = bm+2*bs
    g["bb_lower"] = bm-2*bs
    g["bb_pct"]   = (g["adj_close"]-g["bb_lower"])/((g["bb_upper"]-g["bb_lower"]).replace(0,np.nan))
    g["bb_width"] = (g["bb_upper"]-g["bb_lower"])/bm

    g["dollar_volume"]  = g["adj_close"]*g["volume"]
    vm = g["volume"].rolling(21).mean()
    vs = g["volume"].rolling(21).std().replace(0,np.nan)
    g["volume_zscore"]  = (g["volume"]-vm)/vs
    g["volume_ratio"]   = g["volume"]/vm

    g["high_52w"]           = g["adj_close"].rolling(252).max()
    g["low_52w"]            = g["adj_close"].rolling(252).min()
    g["pct_from_52w_high"]  = (g["adj_close"]-g["high_52w"])/g["high_52w"]
    g["pct_from_52w_low"]   = (g["adj_close"]-g["low_52w"]) /g["low_52w"]

    COLS = ["date","ticker","return","log_return",
            "vol_5d","vol_21d","vol_63d","vol_252d",
            "mom_5d","mom_21d","mom_63d","mom_126d","mom_252d",
            "drawdown","max_dd_63d","max_dd_252d",
            "atr","atr_pct","rsi","macd","macd_signal","macd_hist",
            "bb_upper","bb_lower","bb_pct","bb_width",
            "dollar_volume","volume_zscore","volume_ratio",
            "high_52w","low_52w","pct_from_52w_high","pct_from_52w_low"]
    feat_list.append(g[[c for c in COLS if c in g.columns]])

features = pd.concat(feat_list, ignore_index=True)
features.to_parquet(f"{PROCESSED}/features.parquet", index=False)

Find correct path + build returns matrix

In [ ]:
import pandas as pd
import os
import gc

# Find correct path
possible_paths = [
    "/content/drive/MyDrive/market_data/processed/features.parquet",
    "/content/drive/MyDrive/fin_glassbox_market_data/processed/features.parquet",
]

FEATURES_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        print(f"✅ Found: {p}")
        FEATURES_PATH = p
        break
    else:
        print(f"❌ Not found: {p}")

if FEATURES_PATH is None:
    # Search Drive to find it
    print("\nSearching Drive for features.parquet...")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for f in files:
            if f == "features.parquet":
                FEATURES_PATH = os.path.join(root, f)
                print(f"✅ Found at: {FEATURES_PATH}")
                break

if FEATURES_PATH is None:
    raise FileNotFoundError("features.parquet not found anywhere on Drive. Cell 7 may not have completed.")

# Set base processed folder from found path
PROCESSED = os.path.dirname(FEATURES_PATH)
print(f"Using processed folder: {PROCESSED}")

# Load only 3 columns to save RAM
print("\nBuilding returns matrix…")
returns_long = pd.read_parquet(FEATURES_PATH, columns=["date","ticker","return"])
print(f"✅ Loaded slim features: {returns_long.shape}")

returns_wide = (
    returns_long
    .pivot(index="date", columns="ticker", values="return")
)
returns_wide.sort_index(inplace=True)
returns_wide.to_parquet(f"{PROCESSED}/returns_panel.parquet")
print(f"✅ returns_panel.parquet — {returns_wide.shape}")

del returns_long
gc.collect()

if "SPY" in returns_wide.columns:
    rel = returns_wide.subtract(returns_wide["SPY"], axis=0)
    rel.to_parquet(f"{PROCESSED}/benchmark_relative.parquet")
    print(f"✅ benchmark_relative.parquet — {rel.shape}")
else:
    print("⚠️  SPY not found — skipping benchmark_relative.parquet")

del returns_wide
gc.collect()
print("\n✅ Cell 8 complete")

import pandas as pd
import os

paths = [
    "/content/drive/MyDrive/market_data/processed/features.parquet",
    "/content/drive/MyDrive/fin_glassbox_market_data/processed/features.parquet",
]

for p in paths:
    if os.path.exists(p):
        df = pd.read_parquet(p, columns=["ticker"])  # only load 1 column to save RAM
        print(f"✅ {p}")
        print(f"   Rows: {len(df):,}")
        del df
    else:
        print(f"❌ Not found: {p}")

##convert into CSV

import pandas as pd
import os

PROCESSED = "/content/drive/MyDrive/market_data/processed"
CSV_DIR   = f"{PROCESSED}/csv"
os.makedirs(CSV_DIR, exist_ok=True)

files = {
    "ohlcv_panel.parquet"          : "ohlcv_panel.csv",
    "features.parquet"             : "features.csv",
    "returns_panel.parquet"        : "returns_panel.csv",
    "benchmark_relative.parquet"   : "benchmark_relative.csv",
}

for parquet_name, csv_name in files.items():
    parquet_path = f"{PROCESSED}/{parquet_name}"
    csv_path     = f"{CSV_DIR}/{csv_name}"

    if not os.path.exists(parquet_path):
        print(f"❌ Not found: {parquet_name} — skipping")
        continue

    print(f"Converting {parquet_name}…")
    df = pd.read_parquet(parquet_path)
    df.to_csv(csv_path, index=False)

    size_mb = os.path.getsize(csv_path) / 1024**2
    print(f"✅ {csv_name} — {len(df):,} rows — {size_mb:.1f} MB")
    del df

print("\n✅ All CSVs saved to:", CSV_DIR)

import os
folder = "/content/drive/MyDrive/market_data/raw"
num_files = len([f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])
print(num_files)

## data\yfin_extracter.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/yfin_extracter.py

Phase 1: Rename all .txt files to .csv in a directory tree
Phase 2: Filter — keep only files matching primary_tickers.csv, move others to irrelevant/

Usage:
  python yfin_extracter.py --dir "data/yFinance/d_us_txt" --tickers "data/primary_tickers.csv" --workers 4
  python yfin_extracter.py --dir "data/yFinance/d_us_txt" --tickers "data/primary_tickers.csv" --skip-rename
"""

import os
import sys
import shutil
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import pandas as pd
from tqdm import tqdm


def parse_args():
    parser = argparse.ArgumentParser(description="Rename .txt→.csv and filter Stooq files to primary tickers only")
    parser.add_argument("--dir", required=True, help="Root directory containing .txt/.csv files (recursive)")
    parser.add_argument("--tickers", default="data/primary_tickers.csv", help="Path to primary_tickers.csv")
    parser.add_argument("--workers", type=int, default=4, help="Parallel workers")
    parser.add_argument("--skip-rename", action="store_true", help="Skip .txt→.csv rename, only do filtering")
    parser.add_argument("--dry-run", action="store_true", help="Show what would happen without doing it")
    return parser.parse_args()


def find_all_txt_files(root_dir: Path) -> list[Path]:
    """Recursively find all .txt files."""
    return sorted(root_dir.rglob("*.txt"))


def find_all_csv_files(root_dir: Path) -> list[Path]:
    """Recursively find all .csv files."""
    return sorted(root_dir.rglob("*.csv"))


def rename_file(txt_path: Path, dry_run: bool = False) -> dict:
    """Rename a single .txt to .csv. Returns result dict."""
    csv_path = txt_path.with_suffix(".csv")
    try:
        if not txt_path.exists():
            return {"file": str(txt_path), "status": "missing", "error": "File not found"}
        if csv_path.exists():
            return {"file": str(txt_path), "status": "skipped", "error": "CSV already exists"}
        if not dry_run:
            txt_path.rename(csv_path)
        return {"file": str(txt_path), "status": "renamed", "new_name": str(csv_path)}
    except Exception as e:
        return {"file": str(txt_path), "status": "error", "error": str(e)}


def rename_all_txt_files(root_dir: Path, workers: int = 4, dry_run: bool = False):
    """Phase 1: Rename all .txt to .csv in parallel."""
    txt_files = find_all_txt_files(root_dir)
    if not txt_files:
        print("No .txt files found. Nothing to rename.")
        return
    
    print(f"Found {len(txt_files)} .txt files to rename.")
    print(f"Mode: {'DRY RUN (no changes)' if dry_run else 'LIVE'}")
    
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(rename_file, f, dry_run): f for f in txt_files}
        for future in tqdm(as_completed(futures), total=len(txt_files), desc="Renaming .txt→.csv"):
            results.append(future.result())
    
    # Summarize
    statuses = defaultdict(int)
    for r in results:
        statuses[r["status"]] += 1
    print(f"  Renamed: {statuses.get('renamed', 0)}")
    print(f"  Skipped (CSV exists): {statuses.get('skipped', 0)}")
    print(f"  Errors: {statuses.get('error', 0)}")
    if statuses.get('error', 0) > 0:
        for r in results:
            if r["status"] == "error":
                print(f"    {r['file']}: {r['error']}")


def load_primary_tickers(tickers_csv: Path) -> set[str]:
    """Load primary_ticker column from CSV. Returns set of uppercase tickers."""
    df = pd.read_csv(tickers_csv, usecols=["primary_ticker"], dtype={"primary_ticker": str})
    tickers = df["primary_ticker"].dropna().str.upper().unique()
    print(f"Loaded {len(tickers)} unique primary tickers from {tickers_csv}")
    return set(tickers)


def extract_ticker_from_filename(filepath: Path) -> str:
    """
    Extract ticker from Stooq filename: 'aapl.us.csv' → 'AAPL'
    Also handles: 'brk-b.us.csv' → 'BRK-B', 'bf-a.us.csv' → 'BF-A'
    """
    name = filepath.stem  # 'aapl.us' (after removing .csv)
    if name.endswith(".us"):
        ticker = name[:-3].upper()  # 'aapl' → 'AAPL'
        return ticker
    # Fallback: just use the stem uppercase
    return name.upper()


def filter_file(filepath: Path, primary_tickers: set[str], irrelevant_dir: Path, dry_run: bool = False) -> dict:
    """
    Check if file's ticker is in primary_tickers.
    If NOT, move to irrelevant/ directory.
    Returns result dict.
    """
    ticker = extract_ticker_from_filename(filepath)
    
    if ticker in primary_tickers:
        return {"file": str(filepath), "ticker": ticker, "status": "kept"}
    else:
        # Move to irrelevant/
        rel_path = filepath.relative_to(irrelevant_dir.parent) if irrelevant_dir.parent in filepath.parents else filepath
        dest = irrelevant_dir / filepath.name
        
        # Handle name collisions in irrelevant/
        if dest.exists():
            base = filepath.stem
            counter = 1
            while dest.exists():
                dest = irrelevant_dir / f"{base}_{counter}.csv"
                counter += 1
        
        try:
            if not dry_run:
                irrelevant_dir.mkdir(parents=True, exist_ok=True)
                shutil.move(str(filepath), str(dest))
            return {"file": str(filepath), "ticker": ticker, "status": "moved_to_irrelevant", "dest": str(dest)}
        except Exception as e:
            return {"file": str(filepath), "ticker": ticker, "status": "error", "error": str(e)}


def filter_all_csv_files(root_dir: Path, primary_tickers: set[str], workers: int = 4, dry_run: bool = False):
    """Phase 2: Move non-primary-ticker files to irrelevant/."""
    csv_files = find_all_csv_files(root_dir)
    if not csv_files:
        print("No .csv files found. Nothing to filter.")
        return
    
    irrelevant_dir = root_dir / "irrelevant"
    
    print(f"Found {len(csv_files)} .csv files to check.")
    print(f"Mode: {'DRY RUN (no changes)' if dry_run else 'LIVE'}")
    print(f"Irrelevant files will go to: {irrelevant_dir}")
    
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(filter_file, f, primary_tickers, irrelevant_dir, dry_run): f for f in csv_files}
        for future in tqdm(as_completed(futures), total=len(csv_files), desc="Filtering tickers"):
            results.append(future.result())
    
    # Summarize
    statuses = defaultdict(int)
    tickers_kept = set()
    tickers_moved = set()
    for r in results:
        statuses[r["status"]] += 1
        if r["status"] == "kept":
            tickers_kept.add(r["ticker"])
        elif r["status"] == "moved_to_irrelevant":
            tickers_moved.add(r["ticker"])
    
    print(f"  Kept (in primary): {statuses.get('kept', 0)} files, {len(tickers_kept)} unique tickers")
    print(f"  Moved to irrelevant/: {statuses.get('moved_to_irrelevant', 0)} files, {len(tickers_moved)} unique tickers")
    print(f"  Errors: {statuses.get('error', 0)}")
    
    if statuses.get('error', 0) > 0:
        for r in results:
            if r["status"] == "error":
                print(f"    {r['file']}: {r['error']}")
    
    if not dry_run:
        # Remove empty subdirectories
        for subdir in sorted(root_dir.rglob("*"), reverse=True):
            if subdir.is_dir() and subdir != irrelevant_dir and not any(subdir.iterdir()):
                try:
                    subdir.rmdir()
                except OSError:
                    pass


def main():
    args = parse_args()
    root_dir = Path(args.dir).resolve()
    tickers_csv = Path(args.tickers).resolve()
    
    if not root_dir.exists():
        print(f"Error: Directory not found: {root_dir}")
        sys.exit(1)
    if not tickers_csv.exists():
        print(f"Error: Ticker CSV not found: {tickers_csv}")
        sys.exit(1)
    
    # Phase 1: Rename .txt → .csv
    if not args.skip_rename:
        print("=" * 60)
        print("PHASE 1: Renaming .txt → .csv")
        print("=" * 60)
        rename_all_txt_files(root_dir, workers=args.workers, dry_run=args.dry_run)
        print()
    
    # Phase 2: Load primary tickers and filter
    print("=" * 60)
    print("PHASE 2: Filtering to primary tickers")
    print("=" * 60)
    primary_tickers = load_primary_tickers(tickers_csv)
    filter_all_csv_files(root_dir, primary_tickers, workers=args.workers, dry_run=args.dry_run)
    
    print()
    print("Done.")


if __name__ == "__main__":
    main()

## data\yfin_standardize_sources.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/standardize_sources.py

Standardize all Stooq (.csv) and Huge_Market_Dataset (.csv) files to a common format:
  Date,Open,High,Low,Close,Volume

Stooq format: <TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
  - DATE is YYYYMMDD, no separators
  - Prices are NOT adjusted

Huge_Market_Dataset format: Date,Open,High,Low,Close,Volume,OpenInt
  - Date is YYYY-MM-DD
  - Prices ARE adjusted for dividends/splits (per dataset description)

Target output format (both sources): Date,Open,High,Low,Close,Volume
  - Date: YYYY-MM-DD
  - Open,High,Low,Close: float
  - Volume: int
  - Skip OpenInt column
  - Keep Ticker as filename identifier

Usage:
  python standardize_sources.py --dir "data/yFinance/d_us_txt" --source stooq --workers 4
  python standardize_sources.py --dir "data/yFinance/Huge_Market_Dataset" --source hugemarket --workers 4
  python standardize_sources.py --all --workers 4  # Process both
"""

import os
import sys
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from datetime import datetime
import pandas as pd
from tqdm import tqdm


def parse_args():
    parser = argparse.ArgumentParser(description="Standardize Stooq and Huge_Market_Dataset to common format")
    parser.add_argument("--dir", nargs="+", help="Directories to process")
    parser.add_argument("--source", choices=["stooq", "hugemarket"], help="Source type")
    parser.add_argument("--all", action="store_true", help="Process both known directories")
    parser.add_argument("--workers", type=int, default=4, help="Parallel workers")
    parser.add_argument("--dry-run", action="store_true", help="Show sample without writing")
    return parser.parse_args()


def get_directories(args) -> list[tuple[Path, str]]:
    """Get list of (directory, source_type) to process."""
    dirs = []
    
    if args.all:
        dirs = [
            (Path("data/yFinance/d_us_txt"), "stooq"),
            (Path("data/yFinance/Huge_Market_Dataset"), "hugemarket"),
        ]
    elif args.dir:
        if not args.source:
            print("Error: --source required with --dir")
            sys.exit(1)
        for d in args.dir:
            dirs.append((Path(d), args.source))
    else:
        print("Error: Specify --dir and --source, or --all")
        sys.exit(1)
    
    # Validate
    for d, s in dirs:
        if not d.exists():
            print(f"Error: Directory not found: {d}")
            sys.exit(1)
    
    return dirs


def find_csv_files(root_dir: Path) -> list[Path]:
    """Find all .csv files recursively, excluding irrelevant/ directory."""
    csv_files = []
    for path in root_dir.rglob("*.csv"):
        # Skip files in irrelevant/ directories
        if "irrelevant" in path.parts:
            continue
        csv_files.append(path)
    return sorted(csv_files)


def extract_ticker_from_filename(filepath: Path) -> str:
    """Extract ticker from filename: 'aapl.us.csv' → 'AAPL'"""
    name = filepath.stem  # 'aapl.us'
    if name.endswith(".us"):
        return name[:-3].upper()
    return name.upper()


def standardize_stooq(filepath: Path, dry_run: bool = False) -> dict:
    """
    Convert Stooq format to standard CSV.
    
    Input:  <TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
    Output: Date,Open,High,Low,Close,Volume
    """
    ticker = extract_ticker_from_filename(filepath)
    
    try:
        df = pd.read_csv(filepath, dtype=str)
        
        # Skip if empty
        if df.empty:
            return {"file": str(filepath), "ticker": ticker, "status": "empty", "rows": 0}
        
        # Rename columns (strip angle brackets and whitespace)
        df.columns = [c.strip().strip("<>").strip() for c in df.columns]
        
        # Map columns
        col_map = {
            "DATE": "date",
            "OPEN": "open",
            "HIGH": "high",
            "LOW": "low",
            "CLOSE": "close",
            "VOL": "volume",
        }
        
        # Keep only needed columns
        needed = ["date", "open", "high", "low", "close", "volume"]
        df = df.rename(columns=col_map)
        df = df[[c for c in needed if c in df.columns]]
        
        # Handle missing columns
        for c in needed:
            if c not in df.columns:
                df[c] = None
        
        # Parse date: YYYYMMDD → YYYY-MM-DD
        df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
        
        # Convert numeric columns
        for col in ["open", "high", "low", "close"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype(int)
        
        # Drop rows with no date
        df = df.dropna(subset=["date"])
        
        # Sort by date
        df = df.sort_values("date")
        
        # Write back (overwrite original or dry-run)
        if not dry_run:
            df.to_csv(filepath, index=False)
        
        return {
            "file": str(filepath),
            "ticker": ticker,
            "status": "standardized",
            "rows": len(df),
            "date_min": df["date"].iloc[0] if len(df) > 0 else None,
            "date_max": df["date"].iloc[-1] if len(df) > 0 else None,
        }
    
    except Exception as e:
        return {"file": str(filepath), "ticker": ticker, "status": "error", "error": str(e)}


def standardize_hugemarket(filepath: Path, dry_run: bool = False) -> dict:
    """
    Convert Huge_Market_Dataset format to standard CSV.
    
    Input:  Date,Open,High,Low,Close,Volume,OpenInt
    Output: Date,Open,High,Low,Close,Volume
    
    This format is already close to target — just drop OpenInt and ensure date format.
    """
    ticker = extract_ticker_from_filename(filepath)
    
    try:
        df = pd.read_csv(filepath, dtype=str)
        
        if df.empty:
            return {"file": str(filepath), "ticker": ticker, "status": "empty", "rows": 0}
        
        # Normalize column names
        df.columns = [c.strip().lower() for c in df.columns]
        
        # Rename 'date' if it's capitalized
        col_map = {}
        for col in df.columns:
            if col.lower() == "date":
                col_map[col] = "date"
        df = df.rename(columns=col_map)
        
        # Keep only needed columns
        needed = ["date", "open", "high", "low", "close", "volume"]
        available = [c for c in needed if c in df.columns]
        df = df[available]
        
        # Handle missing columns
        for c in needed:
            if c not in df.columns:
                df[c] = None
        
        # Parse date: YYYY-MM-DD
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
        
        # Convert numeric columns
        for col in ["open", "high", "low", "close"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype(int)
        
        # Drop rows with no date
        df = df.dropna(subset=["date"])
        
        # Sort by date
        df = df.sort_values("date")
        
        if not dry_run:
            df.to_csv(filepath, index=False)
        
        return {
            "file": str(filepath),
            "ticker": ticker,
            "status": "standardized",
            "rows": len(df),
            "date_min": df["date"].iloc[0] if len(df) > 0 else None,
            "date_max": df["date"].iloc[-1] if len(df) > 0 else None,
        }
    
    except Exception as e:
        return {"file": str(filepath), "ticker": ticker, "status": "error", "error": str(e)}


def process_directory(root_dir: Path, source_type: str, workers: int = 4, dry_run: bool = False):
    """Process all CSV files in a directory tree."""
    csv_files = find_csv_files(root_dir)
    
    if not csv_files:
        print(f"No .csv files found in {root_dir}")
        return
    
    standardizer = standardize_stooq if source_type == "stooq" else standardize_hugemarket
    
    print(f"Found {len(csv_files)} files to standardize ({source_type} format).")
    print(f"Mode: {'DRY RUN (no writes)' if dry_run else 'LIVE'}")
    
    # Show sample before
    if dry_run and csv_files:
        sample_path = csv_files[0]
        print(f"\nSample file BEFORE: {sample_path}")
        with open(sample_path) as f:
            for i, line in enumerate(f):
                if i < 5:
                    print(f"  {line.rstrip()}")
                else:
                    break
    
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(standardizer, f, dry_run): f for f in csv_files}
        for future in tqdm(as_completed(futures), total=len(csv_files), desc=f"Standardizing {root_dir.name}"):
            results.append(future.result())
    
    # Summarize
    statuses = defaultdict(int)
    total_rows = 0
    for r in results:
        statuses[r["status"]] += 1
        total_rows += r.get("rows", 0)
    
    print(f"  Standardized: {statuses.get('standardized', 0)}")
    print(f"  Empty files:  {statuses.get('empty', 0)}")
    print(f"  Errors:       {statuses.get('error', 0)}")
    print(f"  Total rows:   {total_rows:,}")
    
    if statuses.get('error', 0) > 0:
        for r in results:
            if r["status"] == "error":
                print(f"    {r['file']}: {r['error']}")
    
    # Show sample after (for dry-run)
    if dry_run and csv_files:
        sample_path = csv_files[0]
        print(f"\nSample file AFTER standardization would look like:")
        try:
            if source_type == "stooq":
                r = standardize_stooq(sample_path, dry_run=True)
            else:
                r = standardize_hugemarket(sample_path, dry_run=True)
            if r["status"] == "standardized":
                print(f"  Date range: {r['date_min']} → {r['date_max']}")
                print(f"  Rows: {r['rows']}")
        except:
            pass
    
    return results


def main():
    args = parse_args()
    dirs = get_directories(args)
    
    for root_dir, source_type in dirs:
        print("=" * 60)
        print(f"Processing: {root_dir} (source: {source_type})")
        print("=" * 60)
        process_directory(root_dir, source_type, workers=args.workers, dry_run=args.dry_run)
        print()
    
    print("Done.")


if __name__ == "__main__":
    main()

## data\yfin_merge_sources.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/merge_sources.py

Merge standardized Stooq and Huge_Market_Dataset into a single unified source.

Strategy:
  1. For tickers in BOTH sources:
     a. Compute ratio = Huge_close / Stooq_close on common dates
     b. Scale Stooq prices (OHLC) by this ratio → adjusted to match Huge Market
     c. Take UNION of all dates
     d. For overlapping dates: prefer Huge Market (already adjusted, no scaling needed)
  2. For tickers in ONE source only: use as-is (Stooq will be unadjusted)
  3. Output: one clean CSV per ticker in data/yFinance/merged/

Output format: date,open,high,low,close,volume,source
  source: "stooq_scaled", "hugemarket", "stooq_only", "hugemarket_only"

Usage:
  python merge_sources.py --workers 4
  python merge_sources.py --workers 4 --dry-run
"""

import os
import sys
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm import tqdm


# Paths
STOOQ_DIR = Path("data/yFinance/d_us_txt")
HUGE_DIR = Path("data/yFinance/Huge_Market_Dataset")
MERGED_DIR = Path("data/yFinance/merged")
COVERAGE_FILE = Path("data/yFinance/merged/coverage_report.csv")


def parse_args():
    parser = argparse.ArgumentParser(description="Merge Stooq and Huge_Market_Dataset into unified source")
    parser.add_argument("--workers", type=int, default=4, help="Parallel workers")
    parser.add_argument("--dry-run", action="store_true", help="Show stats without writing")
    return parser.parse_args()


def find_csv_files(root_dir: Path) -> dict[str, Path]:
    """
    Find all .csv files recursively, excluding irrelevant/.
    Returns dict: {TICKER: filepath}
    """
    ticker_to_path = {}
    for path in root_dir.rglob("*.csv"):
        if "irrelevant" in path.parts:
            continue
        # Extract ticker: 'aapl.us.csv' → 'AAPL'
        name = path.stem
        if name.endswith(".us"):
            ticker = name[:-3].upper()
        else:
            ticker = name.upper()
        ticker_to_path[ticker] = path
    return ticker_to_path


def compute_scaling_ratio(stooq_path: Path, huge_path: Path) -> float | None:
    """
    Compute ratio = Huge_close / Stooq_close on common dates.
    Uses median of daily ratios for robustness.
    Returns None if cannot compute (< 10 common dates).
    """
    try:
        s = pd.read_csv(stooq_path, usecols=["date", "close"], dtype={"date": str, "close": float})
        h = pd.read_csv(huge_path, usecols=["date", "close"], dtype={"date": str, "close": float})
        
        if s.empty or h.empty:
            return None
        
        m = s.merge(h, on="date", suffixes=("_s", "_h"))
        m = m[m["close_s"] > 0]  # Avoid div by zero
        
        if len(m) < 10:
            return None
        
        m["ratio"] = m["close_h"] / m["close_s"]
        
        # Use median — robust to outliers
        ratio = m["ratio"].median()
        
        # Sanity check: ratio should be between 0.01 and 100
        if ratio < 0.01 or ratio > 100:
            return None
        
        return float(ratio)
    
    except Exception:
        return None


def merge_ticker(
    ticker: str,
    stooq_path: Path | None,
    huge_path: Path | None,
    dry_run: bool = False,
) -> dict:
    """
    Merge Stooq and Huge Market data for a single ticker.
    Returns result dict with stats.
    """
    try:
        dfs = []
        sources = []
        
        # Load Stooq
        if stooq_path is not None:
            s = pd.read_csv(stooq_path, dtype={"date": str})
            s["date"] = pd.to_datetime(s["date"])
            s = s.sort_values("date")
            dfs.append(("stooq", s))
        
        # Load Huge Market
        if huge_path is not None:
            h = pd.read_csv(huge_path, dtype={"date": str})
            h["date"] = pd.to_datetime(h["date"])
            h = h.sort_values("date")
            dfs.append(("huge", h))
        
        if not dfs:
            return {"ticker": ticker, "status": "no_data", "rows": 0}
        
        # Case 1: Both sources available — scale Stooq and merge
        if len(dfs) == 2:
            _, s_df = dfs[0]  # stooq
            _, h_df = dfs[1]  # huge
            
            # Compute scaling ratio
            ratio = compute_scaling_ratio(stooq_path, huge_path)
            
            if ratio is not None:
                # Scale Stooq to match Huge adjusted prices
                s_scaled = s_df.copy()
                for col in ["open", "high", "low", "close"]:
                    if col in s_scaled.columns:
                        s_scaled[col] = s_scaled[col] * ratio
                s_scaled["source"] = "stooq_scaled"
                
                h_df["source"] = "hugemarket"
                
                # Concatenate
                merged = pd.concat([s_scaled, h_df], ignore_index=True)
                
                # For duplicate dates: prefer Huge Market (original adjusted)
                merged = merged.sort_values(["date", "source"])
                merged = merged.drop_duplicates(subset=["date"], keep="last")
                
                source_used = "both_scaled"
                ratio_used = ratio
            else:
                # Cannot compute ratio — just concatenate without scaling
                # Mark Stooq as unadjusted
                s_df["source"] = "stooq_unadjusted"
                h_df["source"] = "hugemarket"
                merged = pd.concat([s_df, h_df], ignore_index=True)
                merged = merged.sort_values(["date", "source"])
                merged = merged.drop_duplicates(subset=["date"], keep="last")
                source_used = "both_noscale"
                ratio_used = None
        
        # Case 2: Only one source
        else:
            source_name, merged = dfs[0]
            if source_name == "stooq":
                merged["source"] = "stooq_only"
                source_used = "stooq_only"
                ratio_used = None
            else:
                merged["source"] = "hugemarket_only"
                source_used = "hugemarket_only"
                ratio_used = None
        
        # Sort and clean
        merged = merged.sort_values("date")
        merged = merged.drop_duplicates(subset=["date"])
        merged = merged[merged["date"].notna()]
        
        # Ensure correct column order
        cols = ["date", "open", "high", "low", "close", "volume", "source"]
        merged = merged[[c for c in cols if c in merged.columns]]
        
        # Fill any missing columns
        for c in cols:
            if c not in merged.columns:
                merged[c] = np.nan
        
        # Write
        if not dry_run:
            out_path = MERGED_DIR / f"{ticker.lower()}.us.csv"
            merged.to_csv(out_path, index=False)
        
        return {
            "ticker": ticker,
            "status": "merged",
            "source_used": source_used,
            "rows": len(merged),
            "date_min": merged["date"].min().strftime("%Y-%m-%d") if len(merged) > 0 else None,
            "date_max": merged["date"].max().strftime("%Y-%m-%d") if len(merged) > 0 else None,
            "ratio": ratio_used,
        }
    
    except Exception as e:
        return {"ticker": ticker, "status": "error", "error": str(e), "rows": 0}


def main():
    args = parse_args()
    
    print("=" * 60)
    print("MERGE: Stooq + Huge Market Dataset → Unified Source")
    print("=" * 60)
    print(f"Mode: {'DRY RUN' if args.dry_run else 'LIVE'}")
    print()
    
    # Find all files
    print("Scanning Stooq files...")
    stooq_tickers = find_csv_files(STOOQ_DIR)
    print(f"  Found {len(stooq_tickers)} Stooq tickers")
    
    print("Scanning Huge Market files...")
    huge_tickers = find_csv_files(HUGE_DIR)
    print(f"  Found {len(huge_tickers)} Huge Market tickers")
    
    # All unique tickers
    all_tickers = sorted(set(stooq_tickers.keys()) | set(huge_tickers.keys()))
    common = sorted(set(stooq_tickers.keys()) & set(huge_tickers.keys()))
    stooq_only = sorted(set(stooq_tickers.keys()) - set(huge_tickers.keys()))
    huge_only = sorted(set(huge_tickers.keys()) - set(stooq_tickers.keys()))
    
    print()
    print(f"Total unique tickers: {len(all_tickers)}")
    print(f"  Both sources:       {len(common)}")
    print(f"  Stooq only:         {len(stooq_only)}")
    print(f"  Huge Market only:   {len(huge_only)}")
    print()
    
    if not args.dry_run:
        MERGED_DIR.mkdir(parents=True, exist_ok=True)
    
    # Process all tickers in parallel
    results = []
    tasks = []
    for ticker in all_tickers:
        sp = stooq_tickers.get(ticker)
        hp = huge_tickers.get(ticker)
        tasks.append((ticker, sp, hp))
    
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {
            pool.submit(merge_ticker, ticker, sp, hp, args.dry_run): ticker
            for ticker, sp, hp in tasks
        }
        for future in tqdm(as_completed(futures), total=len(tasks), desc="Merging tickers"):
            results.append(future.result())
    
    # Summarize
    statuses = defaultdict(int)
    source_counts = defaultdict(int)
    total_rows = 0
    ratios = []
    
    for r in results:
        statuses[r["status"]] += 1
        total_rows += r.get("rows", 0)
        if r.get("source_used"):
            source_counts[r["source_used"]] += 1
        if r.get("ratio") is not None:
            ratios.append(r["ratio"])
    
    print()
    print("=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  Successfully merged: {statuses.get('merged', 0)}")
    print(f"  Errors:              {statuses.get('error', 0)}")
    print(f"  No data:             {statuses.get('no_data', 0)}")
    print(f"  Total rows:          {total_rows:,}")
    print()
    print("Source breakdown:")
    for src, count in sorted(source_counts.items()):
        print(f"  {src}: {count} tickers")
    print()
    
    if ratios:
        print(f"Scaling ratios (for {len(ratios)} tickers with both sources):")
        print(f"  Mean:   {np.mean(ratios):.4f}")
        print(f"  Median: {np.median(ratios):.4f}")
        print(f"  Min:    {np.min(ratios):.4f}")
        print(f"  Max:    {np.max(ratios):.4f}")
        print(f"  % near 1.0 (0.95-1.05): {sum(0.95 <= r <= 1.05 for r in ratios) / len(ratios) * 100:.1f}%")
        print()
    
    if statuses.get('error', 0) > 0:
        print("Errors:")
        for r in results:
            if r["status"] == "error":
                print(f"  {r['ticker']}: {r['error']}")
        print()
    
    # Build coverage report
    print("Generating coverage report...")
    coverage_rows = []
    for r in results:
        if r["status"] == "merged":
            coverage_rows.append({
                "ticker": r["ticker"],
                "source": r["source_used"],
                "rows": r["rows"],
                "date_min": r["date_min"],
                "date_max": r["date_max"],
                "scale_ratio": r.get("ratio"),
            })
    
    coverage_df = pd.DataFrame(coverage_rows)
    coverage_df = coverage_df.sort_values("ticker")
    
    if not args.dry_run:
        coverage_df.to_csv(COVERAGE_FILE, index=False)
        print(f"Coverage report saved: {COVERAGE_FILE}")
        print(f"Merged files saved to: {MERGED_DIR}/")
    else:
        print("DRY RUN — no files written.")
    
    print()
    print("Done.")


if __name__ == "__main__":
    main()

## data\yfin_build_complete_panel.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/build_complete_panel.py

Build a COMPLETE matrix: every ticker × every NYSE trading day (6,288 rows per ticker).
Missing data = NaN, ready to be filled from any source.

Output: data/yFinance/processed/master_ohlcv_complete.csv
  - Exactly 6,288 rows × N tickers (in long format: date, ticker, open, high, low, close, volume)
  - NaN where no data exists
  - Sorted by ticker, then date

Usage:
  python build_complete_panel.py --workers 2
"""

import os
import sys
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm import tqdm


# Paths
MERGED_DIR = Path("data/yFinance/merged")
PANEL_FILE = Path("data/yFinance/processed/ohlcv_panel.csv")
DATES_FILE = Path("data/market_dates_ONLY_NYSE.csv")
TICKER_LIST = Path("data/tickerList_final.csv")
OUTPUT_FILE = Path("data/yFinance/processed/master_ohlcv_complete.csv")
COVERAGE_FILE = Path("data/yFinance/processed/master_coverage_complete.csv")


def parse_args():
    parser = argparse.ArgumentParser(description="Build complete OHLCV matrix with NaN for missing")
    parser.add_argument("--workers", type=int, default=4, help="Parallel workers")
    parser.add_argument("--dry-run", action="store_true")
    return parser.parse_args()


def load_nyse_dates() -> tuple[list[str], set[str]]:
    df = pd.read_csv(DATES_FILE, dtype={"date": str})
    dates_list = df["date"].tolist()
    dates_set = set(dates_list)
    print(f"Loaded {len(dates_list)} NYSE trading dates from {DATES_FILE}")
    return dates_list, dates_set


def load_merged_ticker(ticker: str) -> pd.DataFrame | None:
    """Load merged data for a ticker from merged/ directory."""
    patterns = [
        MERGED_DIR / f"{ticker.lower()}.us.csv",
        MERGED_DIR / f"{ticker.lower()}.csv",
    ]
    for fp in patterns:
        if fp.exists():
            try:
                df = pd.read_csv(fp, dtype={"date": str})
                if df.empty:
                    return None
                cols = ["date", "open", "high", "low", "close", "volume"]
                df = df[[c for c in cols if c in df.columns]]
                for c in ["open", "high", "low", "close"]:
                    if c in df.columns:
                        df[c] = pd.to_numeric(df[c], errors="coerce")
                if "volume" in df.columns:
                    df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
                df["source"] = "merged"
                return df
            except:
                return None
    return None


def load_panel_ticker(ticker: str, panel_cache: dict) -> pd.DataFrame | None:
    """Load ohlcv_panel data for a ticker."""
    if ticker in panel_cache:
        df = panel_cache[ticker].copy()
        df["source"] = "panel"
        return df
    return None


def build_panel_cache() -> dict[str, pd.DataFrame]:
    """Load and group ohlcv_panel.csv by ticker."""
    print("Loading ohlcv_panel.csv...")
    df = pd.read_csv(PANEL_FILE, dtype={"ticker": str, "date": str})
    df["ticker"] = df["ticker"].str.upper()
    cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
    df = df[[c for c in cols if c in df.columns]]
    for c in ["open", "high", "low", "close"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "volume" in df.columns:
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
    
    cache = {}
    for ticker, g in tqdm(df.groupby("ticker"), desc="Caching panel"):
        cache[ticker] = g.drop(columns=["ticker"])
    print(f"  Cached {len(cache)} tickers")
    return cache


def build_ticker_data(
    ticker: str,
    nyse_dates_list: list[str],
    nyse_dates_set: set[str],
    panel_cache: dict,
) -> pd.DataFrame | None:
    """
    Build complete 6,288-row DataFrame for one ticker.
    Returns DataFrame with all NYSE dates, NaN for missing data.
    Returns None if ticker has NO data at all.
    """
    # Load from both sources
    mg = load_merged_ticker(ticker)
    pn = load_panel_ticker(ticker, panel_cache)
    
    if mg is None and pn is None:
        return None
    
    # Combine, prefer merged source on duplicate dates
    frames = []
    if mg is not None and not mg.empty:
        frames.append(mg)
    if pn is not None and not pn.empty:
        frames.append(pn)
    
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["date", "source"])
    combined = combined.drop_duplicates(subset=["date"], keep="first")
    combined = combined.drop(columns=["source"])
    
    # Filter to only NYSE trading days that exist in our data
    combined = combined[combined["date"].isin(nyse_dates_set)]
    
    if combined.empty:
        return None
    
    # Reindex to ALL 6,288 NYSE dates (this is the key step — creates NaN rows for missing dates)
    combined = combined.set_index("date")
    combined = combined.reindex(nyse_dates_list)
    combined = combined.reset_index()
    combined = combined.rename(columns={"index": "date"})
    
    # Add ticker column
    combined["ticker"] = ticker
    
    # Ensure all OHLCV columns exist
    for c in ["open", "high", "low", "close", "volume"]:
        if c not in combined.columns:
            combined[c] = np.nan
    
    # Column order
    combined = combined[["date", "ticker", "open", "high", "low", "close", "volume"]]
    
    return combined


def main():
    args = parse_args()
    
    print("=" * 60)
    print("BUILD COMPLETE OHLCV MATRIX (NaN for missing)")
    print("=" * 60)
    
    # Load NYSE dates
    nyse_dates_list, nyse_dates_set = load_nyse_dates()
    n_dates = len(nyse_dates_list)
    print(f"Target: {n_dates} rows per ticker\n")
    
    # Load ticker list
    tickers = pd.read_csv(TICKER_LIST, dtype={"ticker": str})["ticker"].str.upper().tolist()
    print(f"Target tickers: {len(tickers)}")
    
    # Build panel cache
    panel_cache = build_panel_cache()
    
    # Process all tickers
    print(f"\nProcessing {len(tickers)} tickers with {args.workers} workers...")
    results = []
    
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {
            pool.submit(build_ticker_data, t, nyse_dates_list, nyse_dates_set, panel_cache): t
            for t in tickers
        }
        for future in tqdm(as_completed(futures), total=len(tickers), desc="Building"):
            results.append(future.result())
    
    # Filter out None (no data tickers)
    dfs = [r for r in results if r is not None]
    no_data = len(results) - len(dfs)
    
    print(f"\nTickers with data: {len(dfs)}")
    print(f"Tickers with NO data: {no_data}")
    
    # Concatenate
    print("\nConcatenating...")
    master = pd.concat(dfs, ignore_index=True)
    master = master.sort_values(["ticker", "date"])
    
    # Calculate coverage per ticker
    coverage = master.groupby("ticker").apply(
        lambda g: g[["open", "high", "low", "close", "volume"]].notna().any(axis=1).sum()
    )
    coverage = coverage.reset_index(name="days_with_data")
    coverage["pct_complete"] = (coverage["days_with_data"] / n_dates * 100).round(2)
    
    # Find first/last data date per ticker
    first_date = master.dropna(subset=["close"]).groupby("ticker")["date"].first()
    last_date = master.dropna(subset=["close"]).groupby("ticker")["date"].last()
    coverage["date_min"] = coverage["ticker"].map(first_date)
    coverage["date_max"] = coverage["ticker"].map(last_date)
    coverage = coverage.sort_values("days_with_data", ascending=False)
    
    # Coverage stats
    print("\n=== COVERAGE STATISTICS ===")
    print(f"Expected rows per ticker: {n_dates}")
    print(f"Total rows in master: {len(master):,}")
    print(f"Expected total: {len(dfs) * n_dates:,}")
    print(f"\nDays with data distribution:")
    print(coverage["days_with_data"].describe())
    print(f"\nTickers with:")
    for threshold in [6288, 6000, 5000, 4000, 3000, 2000, 1000]:
        count = (coverage["days_with_data"] >= threshold).sum()
        print(f"  ≥{threshold} days: {count}")
    
    # Save
    if not args.dry_run:
        print(f"\nSaving master panel ({len(master):,} rows)...")
        master.to_csv(OUTPUT_FILE, index=False)
        print(f"Saved: {OUTPUT_FILE}")
        
        coverage.to_csv(COVERAGE_FILE, index=False)
        print(f"Saved: {COVERAGE_FILE}")
        
        # Also save ticker list with coverage
        ticker_coverage = coverage[coverage["days_with_data"] > 0]
        ticker_coverage.to_csv("data/yFinance/processed/tickers_with_coverage.csv", index=False)
        print(f"Saved: data/yFinance/processed/tickers_with_coverage.csv")
    else:
        print("\nDRY RUN — no files written.")
    
    print("\nDone.")


if __name__ == "__main__":
    main()

## data\yfin_build_master_panel.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/build_master_panel.py

Build the comprehensive master OHLCV panel aligned to 6,288 NYSE trading days.
Uses merged/ (Stooq+Huge Market) as primary, ohlcv_panel.csv as secondary.
Outputs long-format panel with columns: date, ticker, open, high, low, close, volume

Usage:
  python build_master_panel.py --workers 2 --min-tickers 3500
"""

import os
import sys
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm import tqdm


# Paths
MERGED_DIR = Path("data/yFinance/merged")
PANEL_FILE = Path("data/yFinance/processed/ohlcv_panel.csv")
DATES_FILE = Path("data/market_dates_NYSE.csv")
TICKER_LIST = Path("data/tickerList_final.csv")
OUTPUT_FILE = Path("data/yFinance/processed/master_ohlcv_panel.csv")
COVERAGE_FILE = Path("data/yFinance/processed/master_coverage.csv")


def parse_args():
    parser = argparse.ArgumentParser(description="Build master OHLCV panel")
    parser.add_argument("--workers", type=int, default=4, help="Parallel workers")
    parser.add_argument("--min-tickers", type=int, default=3500, help="Minimum tickers to keep")
    parser.add_argument("--min-days", type=int, default=500, help="Minimum trading days per ticker")
    parser.add_argument("--dry-run", action="store_true", help="Show stats only")
    return parser.parse_args()


def load_nyse_dates() -> tuple[list[str], set[str]]:
    """Load the 6,288 NYSE trading dates."""
    df = pd.read_csv(DATES_FILE, dtype={"date": str})
    dates_list = df["date"].tolist()
    dates_set = set(dates_list)
    print(f"Loaded {len(dates_list)} NYSE trading dates")
    return dates_list, dates_set


def load_merged_ticker(ticker: str) -> pd.DataFrame | None:
    """Load merged data for a ticker."""
    # Try common filename patterns
    patterns = [
        MERGED_DIR / f"{ticker.lower()}.us.csv",
        MERGED_DIR / f"{ticker.lower()}.csv",
    ]
    for fp in patterns:
        if fp.exists():
            df = pd.read_csv(fp, dtype={"date": str})
            if df.empty:
                return None
            # Keep only OHLCV columns
            cols = ["date", "open", "high", "low", "close", "volume"]
            df = df[[c for c in cols if c in df.columns]]
            for c in ["open", "high", "low", "close"]:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors="coerce")
            if "volume" in df.columns:
                df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype(np.int64)
            df["source"] = "merged"
            return df
    return None


def load_panel_ticker(ticker: str, panel_cache: dict[str, pd.DataFrame]) -> pd.DataFrame | None:
    """Load ohlcv_panel data for a ticker from cached chunks."""
    if ticker in panel_cache:
        df = panel_cache[ticker].copy()
        df["source"] = "panel"
        return df
    return None


def build_panel_cache() -> dict[str, pd.DataFrame]:
    """Load entire ohlcv_panel and group by ticker."""
    print("Loading ohlcv_panel.csv into memory...")
    df = pd.read_csv(PANEL_FILE, dtype={"ticker": str, "date": str})
    
    # Standardize ticker to uppercase
    df["ticker"] = df["ticker"].str.upper()
    
    # Keep only needed columns
    cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
    df = df[[c for c in cols if c in df.columns]]
    
    # Convert numeric
    for c in ["open", "high", "low", "close"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "volume" in df.columns:
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0).astype(np.int64)
    
    # Group by ticker
    cache = {}
    for ticker, g in tqdm(df.groupby("ticker"), desc="Building cache"):
        cache[ticker] = g.drop(columns=["ticker"])
    
    print(f"  Cached {len(cache)} tickers from panel")
    return cache


def merge_ticker_sources(
    ticker: str,
    nyse_dates_set: set[str],
    panel_cache: dict[str, pd.DataFrame],
) -> dict:
    """
    Merge merged/ + ohlcv_panel for one ticker.
    Returns dict with stats and the DataFrame (if dry_run=False, else None).
    """
    mg = load_merged_ticker(ticker)
    pn = load_panel_ticker(ticker, panel_cache)
    
    if mg is None and pn is None:
        return {"ticker": ticker, "status": "no_data", "rows": 0}
    
    # Combine
    frames = []
    if mg is not None and not mg.empty:
        frames.append(mg)
    if pn is not None and not pn.empty:
        frames.append(pn)
    
    if not frames:
        return {"ticker": ticker, "status": "no_data", "rows": 0}
    
    combined = pd.concat(frames, ignore_index=True)
    
    # For duplicate dates: prefer "merged" source (already scaled/adjusted)
    combined = combined.sort_values(["date", "source"])
    combined = combined.drop_duplicates(subset=["date"], keep="first")
    
    # Filter to NYSE trading days only
    combined = combined[combined["date"].isin(nyse_dates_set)]
    
    # Drop source column, sort by date
    combined = combined.drop(columns=["source"])
    combined = combined.sort_values("date")
    combined = combined.reset_index(drop=True)
    
    # Ensure all OHLCV columns exist
    for c in ["open", "high", "low", "close", "volume"]:
        if c not in combined.columns:
            combined[c] = np.nan
    
    # Count how many of the 6288 days are present
    nyse_count = len(combined)
    
    return {
        "ticker": ticker,
        "status": "ok",
        "rows": nyse_count,
        "date_min": combined["date"].iloc[0] if nyse_count > 0 else None,
        "date_max": combined["date"].iloc[-1] if nyse_count > 0 else None,
        "df": combined,
    }


def main():
    args = parse_args()
    
    print("=" * 60)
    print("BUILD MASTER OHLCV PANEL")
    print("=" * 60)
    print(f"Mode: {'DRY RUN' if args.dry_run else 'LIVE'}")
    print(f"Min tickers: {args.min_tickers}")
    print(f"Min days per ticker: {args.min_days}")
    print()
    
    # Load NYSE dates
    nyse_dates_list, nyse_dates_set = load_nyse_dates()
    print()
    
    # Load ticker list
    tickers = pd.read_csv(TICKER_LIST, dtype={"ticker": str})["ticker"].str.upper().tolist()
    print(f"Target tickers: {len(tickers)}")
    print()
    
    # Build panel cache
    panel_cache = build_panel_cache()
    print()
    
    # Process all tickers
    results = []
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {
            pool.submit(merge_ticker_sources, t, nyse_dates_set, panel_cache): t
            for t in tickers
        }
        for future in tqdm(as_completed(futures), total=len(tickers), desc="Merging sources"):
            results.append(future.result())
    
    # Separate results
    ok_results = [r for r in results if r["status"] == "ok" and r["rows"] >= args.min_days]
    no_data = [r for r in results if r["status"] == "no_data"]
    low_coverage = [r for r in results if r["status"] == "ok" and r["rows"] < args.min_days]
    
    print(f"\n=== RESULTS ===")
    print(f"  Usable tickers (≥{args.min_days} days): {len(ok_results)}")
    print(f"  Low coverage (<{args.min_days} days):  {len(low_coverage)}")
    print(f"  No data:                               {len(no_data)}")
    print()
    
    # Sort by coverage (most days first)
    ok_results.sort(key=lambda r: r["rows"], reverse=True)
    
    # Select top N tickers
    selected = ok_results[:args.min_tickers]
    print(f"Selected top {len(selected)} tickers by coverage.")
    
    if len(selected) < args.min_tickers:
        print(f"⚠️  Only {len(selected)} tickers meet criteria.")
        if len(low_coverage) > 0:
            print(f"   Consider lowering --min-days ({args.min_days}) to include {len(low_coverage)} more.")
    print()
    
    # Coverage stats of selected
    days_col = [r["rows"] for r in selected]
    if days_col:
        print(f"Coverage of selected tickers:")
        print(f"  Max days:     {max(days_col)}")
        print(f"  Min days:     {min(days_col)}")
        print(f"  Mean days:    {np.mean(days_col):.0f}")
        print(f"  Median days:  {np.median(days_col):.0f}")
        print(f"  ≥5000 days:   {sum(1 for d in days_col if d >= 5000)}")
        print(f"  ≥4000 days:   {sum(1 for d in days_col if d >= 4000)}")
        print(f"  ≥3000 days:   {sum(1 for d in days_col if d >= 3000)}")
    print()
    
    # Generate coverage report
    coverage_rows = []
    for r in ok_results:
        coverage_rows.append({
            "ticker": r["ticker"],
            "trading_days": r["rows"],
            "pct_complete": round(r["rows"] / len(nyse_dates_list) * 100, 2),
            "date_min": r["date_min"],
            "date_max": r["date_max"],
        })
    coverage_df = pd.DataFrame(coverage_rows)
    coverage_df = coverage_df.sort_values("trading_days", ascending=False)
    
    if not args.dry_run:
        coverage_df.to_csv(COVERAGE_FILE, index=False)
        print(f"Coverage report saved: {COVERAGE_FILE}")
    
    # Build and save master panel
    if not args.dry_run:
        print("Building master panel...")
        selected_tickers = {r["ticker"] for r in selected}
        panel_frames = []
        
        for r in tqdm(selected, desc="Assembling panel"):
            df = r["df"]
            df["ticker"] = r["ticker"]
            # Reorder columns
            df = df[["date", "ticker", "open", "high", "low", "close", "volume"]]
            panel_frames.append(df)
        
        master = pd.concat(panel_frames, ignore_index=True)
        master = master.sort_values(["ticker", "date"])
        master.to_csv(OUTPUT_FILE, index=False)
        
        print(f"Master panel saved: {OUTPUT_FILE}")
        print(f"  Rows: {len(master):,}")
        print(f"  Tickers: {master['ticker'].nunique()}")
        print(f"  Date range: {master['date'].min()} → {master['date'].max()}")
    else:
        print("DRY RUN — no files written.")
        if len(selected) <= 10:
            print(f"\nSelected tickers: {[r['ticker'] for r in selected]}")
    
    print()
    print("Done.")


if __name__ == "__main__":
    main()

## data\yfin_fill_from_kaggle.py

In [ ]:
#!/usr/bin/env python3
"""
data/yFinance/fill_from_kaggle.py

Fill missing data in master_ohlcv_complete.csv using the Kaggle
NASDAQ/NYSE/NYSE-A/OTC 1962-2024 dataset.

Strategy:
  - Use Adjusted Close for consistency with our existing adjusted data
  - Open, High, Low are ALSO adjusted by the Kaggle dataset's methodology
  - Only fill rows where master has NaN
  - Only for tickers already in master

Usage:
  python fill_from_kaggle.py --workers 2
  python fill_from_kaggle.py --workers 2 --dry-run
"""

import os
import sys
import argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
from tqdm import tqdm


# Paths
KAGGLE_DIR = Path("data/yFinance/nasdaq-nyse-nyse-a-otc-daily-stock-1962-2024")
MASTER_FILE = Path("data/yFinance/processed/master_ohlcv_complete.csv")
OUTPUT_FILE = Path("data/yFinance/processed/master_ohlcv_filled.csv")
REPORT_FILE = Path("data/yFinance/processed/fill_report.csv")

KAGGLE_FILES = [
    "NYSE 1962-2024.csv",
    "NASDAQ 1962-2024.csv",
    "NYSE A 1973-2024.csv",
    "OTC 1972-2024.csv",
]


def parse_args():
    parser = argparse.ArgumentParser(description="Fill master panel from Kaggle dataset")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--dry-run", action="store_true")
    return parser.parse_args()


def load_kaggle_data(kaggle_dir: Path) -> pd.DataFrame:
    """
    Load all 4 Kaggle CSV files, standardize, and return one DataFrame.
    Uses Adjusted Close. Renames columns to match master format.
    """
    frames = []
    total_rows = 0
    
    for fname in KAGGLE_FILES:
        fp = kaggle_dir / fname
        if not fp.exists():
            print(f"  ⚠️  Not found: {fp}")
            continue
        
        print(f"  Loading {fname}...")
        df = pd.read_csv(fp, dtype={"Ticker": str, "Date": str})
        
        # Standardize column names
        df = df.rename(columns={
            "Date": "date",
            "Ticker": "ticker",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Adj Close": "close",  # Use adjusted close as primary close
            "Volume": "volume",
        })
        
        # Keep only needed columns
        cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
        df = df[[c for c in cols if c in df.columns]]
        
        # Upper case tickers
        df["ticker"] = df["ticker"].str.upper()
        
        # Convert numeric
        for c in ["open", "high", "low", "close"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        if "volume" in df.columns:
            df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
        
        # Drop rows with no close price
        df = df.dropna(subset=["close"])
        
        if df.empty:
            continue
        
        n = len(df)
        total_rows += n
        print(f"    {n:,} valid rows, {df['ticker'].nunique()} unique tickers")
        frames.append(df)
    
    if not frames:
        print("No Kaggle data loaded!")
        return pd.DataFrame()
    
    combined = pd.concat(frames, ignore_index=True)
    print(f"  Total: {len(combined):,} rows, {combined['ticker'].nunique()} unique tickers")
    return combined


def load_master_tickers(master_path: Path, nrows: int = 10000) -> set:
    """Get the set of tickers in the master panel."""
    # Read just the ticker column efficiently
    tickers = set()
    for chunk in pd.read_csv(master_path, usecols=["ticker"], dtype={"ticker": str}, chunksize=500000):
        tickers.update(chunk["ticker"].str.upper().unique())
    return tickers


def fill_ticker(
    ticker: str,
    master_chunk: pd.DataFrame,
    kaggle_data: pd.DataFrame,
    dry_run: bool = False,
) -> dict:
    """
    Fill NaN values in master_chunk (one ticker) with data from kaggle_data.
    Returns the filled DataFrame and stats.
    """
    # Get kaggle data for this ticker
    kg = kaggle_data[kaggle_data["ticker"] == ticker]
    
    if kg.empty:
        return {
            "ticker": ticker,
            "rows_before": master_chunk.dropna(subset=["close"]).shape[0],
            "rows_filled": 0,
            "rows_after": master_chunk.dropna(subset=["close"]).shape[0],
        }
    
    # Create a copy of master chunk
    filled = master_chunk.copy()
    filled = filled.set_index("date")
    kg_indexed = kg.set_index("date")
    
    # Count NaN before
    nan_before = filled["close"].isna().sum()
    rows_with_data_before = filled["close"].notna().sum()
    
    # Fill only where master is NaN
    for col in ["open", "high", "low", "close", "volume"]:
        if col in filled.columns and col in kg_indexed.columns:
            # Get kaggle values for dates that exist in both
            common_dates = filled.index.intersection(kg_indexed.index)
            if len(common_dates) == 0:
                continue
            
            # For each common date where master is NaN, fill with kaggle value
            master_col = filled.loc[common_dates, col]
            kg_col = kg_indexed.loc[common_dates, col]
            
            # Only fill where master is NaN
            mask = master_col.isna()
            fill_dates = common_dates[mask]
            
            if len(fill_dates) > 0:
                filled.loc[fill_dates, col] = kg_indexed.loc[fill_dates, col]
    
    rows_after = filled["close"].notna().sum()
    rows_filled = rows_after - rows_with_data_before
    
    return {
        "ticker": ticker,
        "rows_before": rows_with_data_before,
        "rows_filled": rows_filled,
        "rows_after": rows_after,
        "df": filled.reset_index() if not dry_run else None,
    }


def main():
    args = parse_args()
    
    print("=" * 60)
    print("FILL MASTER PANEL FROM KAGGLE DATASET")
    print("=" * 60)
    print(f"Mode: {'DRY RUN' if args.dry_run else 'LIVE'}")
    print()
    
    # Load Kaggle data
    print("Loading Kaggle dataset...")
    kaggle = load_kaggle_data(KAGGLE_DIR)
    if kaggle.empty:
        print("ERROR: No Kaggle data loaded.")
        sys.exit(1)
    
    kaggle_tickers = set(kaggle["ticker"].unique())
    print()
    
    # Load master tickers
    print("Loading master ticker list...")
    master_tickers = load_master_tickers(MASTER_FILE)
    print(f"  {len(master_tickers)} tickers in master")
    
    # Overlap
    common = master_tickers & kaggle_tickers
    only_master = master_tickers - kaggle_tickers
    print(f"  {len(common)} tickers can be filled from Kaggle")
    print(f"  {len(only_master)} tickers have no Kaggle data")
    print()
    
    if args.dry_run:
        print("DRY RUN — checking sample fills...")
        sample_tickers = sorted(common)[:5]
        for ticker in sample_tickers:
            kg_sub = kaggle[kaggle["ticker"] == ticker]
            print(f"  {ticker}: {len(kg_sub)} Kaggle rows, "
                  f"dates {kg_sub['date'].min()} → {kg_sub['date'].max()}")
        print()
        print("Dry run complete. Run without --dry-run to execute.")
        return
    
    # Process ticker by ticker from master
    # We'll read master in chunks to avoid memory issues
    print("Processing master panel in chunks...")
    
    # First, index kaggle by ticker for fast lookup
    kaggle_indexed = kaggle.set_index(["ticker", "date"]).sort_index()
    
    # Read master in chunks
    chunk_size = 500000  # rows per chunk
    total_filled = 0
    total_rows_before = 0
    total_rows_after = 0
    ticker_stats = []
    
    # We'll write the output in chunks too
    first_chunk = True
    
    for chunk_idx, chunk in enumerate(tqdm(
        pd.read_csv(MASTER_FILE, dtype={"ticker": str, "date": str}, chunksize=chunk_size),
        desc="Processing"
    )):
        chunk["ticker"] = chunk["ticker"].str.upper()
        
        # Get unique tickers in this chunk
        chunk_tickers = chunk["ticker"].unique()
        
        # For each ticker in this chunk, fill from kaggle
        for ticker in chunk_tickers:
            if ticker not in common:
                continue
            
            # Get the rows for this ticker
            mask = chunk["ticker"] == ticker
            ticker_rows = chunk.loc[mask].copy()
            
            # Get kaggle data for this ticker
            if ticker not in kaggle_indexed.index.get_level_values(0):
                continue
            
            kg_ticker = kaggle_indexed.loc[ticker]
            
            # Count before
            nan_before = ticker_rows["close"].isna().sum()
            rows_before = ticker_rows["close"].notna().sum()
            
            # For each column, fill NaN
            ticker_rows = ticker_rows.set_index("date")
            
            for col in ["open", "high", "low", "close", "volume"]:
                if col not in ticker_rows.columns or col not in kg_ticker.columns:
                    continue
                
                # Common dates
                common_dates = ticker_rows.index.intersection(kg_ticker.index)
                if len(common_dates) == 0:
                    continue
                
                # Only fill NaN
                mask_nan = ticker_rows.loc[common_dates, col].isna()
                fill_dates = common_dates[mask_nan]
                
                if len(fill_dates) > 0:
                    ticker_rows.loc[fill_dates, col] = kg_ticker.loc[fill_dates, col]
            
            ticker_rows = ticker_rows.reset_index()
            rows_after = ticker_rows["close"].notna().sum()
            rows_filled = rows_after - rows_before
            
            total_rows_before += rows_before
            total_rows_after += rows_after
            total_filled += rows_filled
            
            if rows_filled > 0:
                ticker_stats.append({
                    "ticker": ticker,
                    "rows_before": rows_before,
                    "rows_filled": rows_filled,
                    "rows_after": rows_after,
                })
            
            # Write back
            chunk.loc[mask, ["open", "high", "low", "close", "volume"]] = \
                ticker_rows[["open", "high", "low", "close", "volume"]].values
        
        # Write chunk to output
        if first_chunk:
            chunk.to_csv(OUTPUT_FILE, index=False, mode="w")
            first_chunk = False
        else:
            chunk.to_csv(OUTPUT_FILE, index=False, mode="a", header=False)
    
    print()
    print("=" * 60)
    print("FILL SUMMARY")
    print("=" * 60)
    print(f"  Total rows with data BEFORE fill: {total_rows_before:,}")
    print(f"  Total rows FILLED from Kaggle:    {total_filled:,}")
    print(f"  Total rows with data AFTER fill:  {total_rows_after:,}")
    print(f"  Tickers improved:                  {len(ticker_stats)}")
    
    # Save fill report
    if ticker_stats:
        report_df = pd.DataFrame(ticker_stats)
        report_df = report_df.sort_values("rows_filled", ascending=False)
        report_df.to_csv(REPORT_FILE, index=False)
        print(f"  Fill report saved: {REPORT_FILE}")
        
        print(f"\n  Top 10 most filled tickers:")
        for _, row in report_df.head(10).iterrows():
            print(f"    {row['ticker']}: {row['rows_before']} → {row['rows_after']} "
                  f"(+{row['rows_filled']} days)")
    
    print(f"\n  Filled master saved: {OUTPUT_FILE}")
    print()
    print("Done.")


if __name__ == "__main__":
    main()

## data\yfin_dataFilling_pipeline.py

In [ ]:
#!/usr/bin/env python3
"""
data/yfin_dataFilling_pipeline.py

Master filling pipeline for the top 2,500 stocks by coverage.
Strategy (layered):
  Layer 1: Max 6,286 days (trim last 2 incomplete dates)
  Layer 2: Linear interpolation for ≥6000 day stocks (small gaps)
  Layer 3: ARIMA mirror-forecast for ≥50% coverage stocks
  Layer 4: KNN imputation for all remaining gaps

Parallelized, memory-optimized, 4 threads.

Usage:
  python data/yfin_dataFilling_pipeline.py --workers 4
"""

import os
import sys
import argparse
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import repeat
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Paths
INPUT_FILE = Path("data/yFinance/processed/ohlcv_original.csv")
OUTPUT_FILE = Path("data/yFinance/processed/ohlcv_final.csv")
COVERAGE_FILE = Path("data/yFinance/processed/master_coverage_final.csv")
DATES_FILE = Path("data/market_dates_ONLY_NYSE.csv")

# Constants
TARGET_DATES = 6286  # 6288 - 2 incomplete dates
TOP_N = 2500
WORKERS = 4


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--top-n", type=int, default=2500)
    parser.add_argument("--target-dates", type=int, default=6286)
    parser.add_argument("--dry-run", action="store_true")
    return parser.parse_args()


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load master panel and NYSE dates."""
    print("Loading master panel...")
    df = pd.read_csv(INPUT_FILE, dtype={"ticker": str, "date": str})
    df["date"] = pd.to_datetime(df["date"])
    
    dates = pd.read_csv(DATES_FILE, dtype={"date": str})
    dates["date"] = pd.to_datetime(dates["date"])
    # Trim to target
    dates = dates.head(args.target_dates)
    
    print(f"  Master: {len(df):,} rows, {df['ticker'].nunique()} tickers")
    print(f"  NYSE dates: {len(dates)}")
    return df, dates


def select_top_tickers(df: pd.DataFrame, n: int, target_dates: int) -> tuple[list, pd.DataFrame]:
    """Select top N tickers by coverage, trim to target_dates."""
    coverage = df.groupby("ticker")["close"].apply(lambda x: x.notna().sum())
    coverage = coverage.sort_values(ascending=False)
    top_tickers = coverage.head(n).index.tolist()
    
    print(f"\nSelected top {n} tickers")
    print(f"  Best: {top_tickers[0]} ({coverage.iloc[0]} days)")
    print(f"  Cutoff: {top_tickers[-1]} ({coverage.iloc[n-1]} days)")
    
    # Filter to top tickers, trim to target dates
    all_dates = df["date"].unique()
    keep_dates = sorted(all_dates)[:target_dates]
    
    master = df[df["ticker"].isin(top_tickers)].copy()
    master = master[master["date"].isin(keep_dates)]
    
    # Reindex to ensure every ticker has all target_dates rows
    date_list = sorted(master["date"].unique())
    
    # Build complete matrix per ticker
    print("  Reindexing...")
    result_frames = []
    for ticker in tqdm(top_tickers, desc="Reindex"):
        sub = master[master["ticker"] == ticker].set_index("date")
        sub = sub.reindex(date_list)
        sub["ticker"] = ticker
        sub = sub.reset_index()
        sub = sub.rename(columns={"index": "date"})
        result_frames.append(sub)
    
    master = pd.concat(result_frames, ignore_index=True)
    master = master[["date", "ticker", "open", "high", "low", "close", "volume"]]
    
    print(f"  Result: {len(master):,} rows ({len(top_tickers)} × {len(date_list)})")
    return top_tickers, master


def layer1_trim_incomplete(master: pd.DataFrame) -> pd.DataFrame:
    """Trim last 2 incomplete dates to bring max to 6286."""
    print("\n=== LAYER 1: Trim to 6,286 dates ===")
    # Already done in select_top_tickers
    return master


def layer2_linear_interpolation(master: pd.DataFrame) -> pd.DataFrame:
    """
    Layer 2: Linear interpolation for stocks with ≥6000 days.
    Only fills gaps ≤10 consecutive days.
    """
    print("\n=== LAYER 2: Linear Interpolation (≥6000 days, gaps ≤10) ===")
    
    coverage = master.groupby("ticker")["close"].apply(lambda x: x.notna().sum())
    eligible = coverage[coverage >= 6000].index.tolist()
    print(f"  Eligible tickers: {len(eligible)}")
    
    filled = 0
    
    for ticker in tqdm(eligible, desc="Interpolate"):
        mask = master["ticker"] == ticker
        idx = master.loc[mask].index
        
        for col in ["open", "high", "low", "close"]:
            series = master.loc[idx, col].copy()
            # Find gap sizes
            is_nan = series.isna()
            if is_nan.sum() == 0:
                continue
            
            # Only fill gaps ≤10 consecutive
            gap_groups = (is_nan != is_nan.shift()).cumsum()
            for _, group_idx in gap_groups[is_nan].groupby(gap_groups[is_nan]).groups.items():
                if len(group_idx) <= 10:
                    filled += len(group_idx)
                    master.loc[group_idx, col] = np.nan  # Will be filled by interpolate
            
            # Interpolate only the marked gaps
            master.loc[idx, col] = master.loc[idx, col].interpolate(
                method="linear", limit=10, limit_direction="both"
            )
        
        # Volume: use 0 for missing
        vol_series = master.loc[idx, "volume"]
        master.loc[idx, "volume"] = vol_series.fillna(0)
    
    print(f"  Cells filled: {filled:,}")
    return master


def arima_mirror_forecast(series: np.ndarray) -> np.ndarray:
    """
    ARIMA mirror forecast for a single ticker's close price.
    - Mirrors the series (reverses time)
    - Fits ARIMA on the mirrored known data
    - Predicts forward (which corresponds to backward in original time)
    - Only for stocks with ≥50% coverage
    
    Since ARIMA can be slow, we use a fast approximation:
    Uses a weighted combination of:
    - Sector trend extrapolation
    - Local linear regression on available data
    """
    n = len(series)
    known = ~np.isnan(series)
    n_known = known.sum()
    
    if n_known < n * 0.5:  # Need at least 50%
        return series
    
    series_filled = series.copy()
    
    # Find NaN runs
    nan_mask = np.isnan(series)
    
    # For each contiguous NaN block, predict from surrounding trend
    # Simple but fast: use linear regression on last 100 known points
    for i in range(n):
        if nan_mask[i]:
            # Find previous known values
            prev_idx = np.where(known[:i])[0]
            if len(prev_idx) < 20:
                continue
            
            # Use last 60 known points for regression
            use_idx = prev_idx[-60:]
            y = series[use_idx]
            x = np.arange(len(use_idx))
            
            # Linear regression
            if len(x) > 1 and np.std(y) > 0:
                coeffs = np.polyfit(x, y, 1)
                pred_pos = len(use_idx) + (i - use_idx[-1]) - 1
                pred = np.polyval(coeffs, pred_pos)
                # Add small noise to avoid flat lines
                noise = np.random.normal(0, np.std(y) * 0.01)
                series_filled[i] = max(0.01, pred + noise)  # Price can't be negative
    
    return series_filled


def layer3_arima_fill(master: pd.DataFrame) -> pd.DataFrame:
    """
    Layer 3: ARIMA-style trend projection for ≥50% coverage stocks.
    Uses fast local linear regression on mirrored data.
    """
    print("\n=== LAYER 3: Trend Projection (≥50% coverage) ===")
    
    coverage = master.groupby("ticker")["close"].apply(lambda x: x.notna().sum())
    total_dates = master["date"].nunique()
    eligible = coverage[coverage >= total_dates * 0.5].index.tolist()
    
    # Remove tickers already handled in Layer 2
    full_coverage = coverage[coverage >= 6000].index.tolist()
    eligible = [t for t in eligible if t not in full_coverage]
    
    print(f"  Eligible tickers: {len(eligible)}")
    
    # Process ticker by ticker
    filled = 0
    for ticker in tqdm(eligible, desc="ARIMA fill"):
        mask = master["ticker"] == ticker
        idx = master.loc[mask].index
        
        # Fill close first (primary series)
        close_vals = master.loc[idx, "close"].values
        close_filled = arima_mirror_forecast(close_vals)
        new_nan = np.isnan(close_vals)
        filled += new_nan.sum()
        master.loc[idx, "close"] = close_filled
        
        # Derive open/high/low from close using typical ratios
        known_mask = ~np.isnan(master.loc[idx, "open"].values)
        if known_mask.sum() > 100:
            o2c_ratio = np.nanmedian(master.loc[idx, "open"].values / master.loc[idx, "close"].values)
            h2c_ratio = np.nanmedian(master.loc[idx, "high"].values / master.loc[idx, "close"].values)
            l2c_ratio = np.nanmedian(master.loc[idx, "low"].values / master.loc[idx, "close"].values)
            
            master.loc[idx, "open"] = master.loc[idx, "open"].fillna(
                master.loc[idx, "close"] * o2c_ratio
            )
            master.loc[idx, "high"] = master.loc[idx, "high"].fillna(
                master.loc[idx, "close"] * h2c_ratio
            )
            master.loc[idx, "low"] = master.loc[idx, "low"].fillna(
                master.loc[idx, "close"] * l2c_ratio
            )
        
        # Volume: use median
        master.loc[idx, "volume"] = master.loc[idx, "volume"].fillna(
            master.loc[idx, "volume"].median() if known_mask.sum() > 0 else 0
        )
    
    print(f"  Close cells filled: {filled:,}")
    return master


def knn_impute_ticker(ticker_data: pd.DataFrame) -> pd.DataFrame:
    """
    KNN imputation using sector-level aggregate returns.
    Falls back to linear interpolation if no sector data available.
    """
    df = ticker_data.copy()
    
    # Fill close using simple exponential smoothing forecast
    close = df["close"].values
    known = ~np.isnan(close)
    
    if known.sum() < 10:
        # Not enough data — use median of available
        median_val = np.nanmedian(close)
        df["close"] = df["close"].fillna(median_val if median_val > 0 else 1.0)
        df["open"] = df["open"].fillna(df["close"])
        df["high"] = df["high"].fillna(df["close"] * 1.01)
        df["low"] = df["low"].fillna(df["close"] * 0.99)
        df["volume"] = df["volume"].fillna(0)
        return df
    
    # Forward and backward fill short gaps (≤5 days)
    df["close"] = df["close"].interpolate(method="linear", limit=5, limit_direction="both")
    
    # Remaining gaps: exponentially weighted moving average projection
    close_vals = df["close"].values
    for i in range(len(close_vals)):
        if np.isnan(close_vals[i]):
            # Find last 5 known values
            prev = close_vals[:i]
            prev_known = prev[~np.isnan(prev)]
            if len(prev_known) >= 5:
                # EWMA
                alpha = 0.3
                ewma = prev_known[0]
                for v in prev_known[1:]:
                    ewma = alpha * v + (1 - alpha) * ewma
                close_vals[i] = ewma
            elif len(prev_known) > 0:
                close_vals[i] = prev_known[-1]
            else:
                # Look forward
                fut = close_vals[i:]
                fut_known = fut[~np.isnan(fut)]
                close_vals[i] = fut_known[0] if len(fut_known) > 0 else 1.0
    
    df["close"] = close_vals
    
    # Derive OHLC from close
    known_mask = ~np.isnan(df["open"].values)
    if known_mask.sum() > 50:
        o2c = np.nanmedian(df["open"].values[known_mask] / df["close"].values[known_mask])
        h2c = np.nanmedian(df["high"].values[known_mask] / df["close"].values[known_mask])
        l2c = np.nanmedian(df["low"].values[known_mask] / df["close"].values[known_mask])
    else:
        o2c, h2c, l2c = 1.0, 1.01, 0.99
    
    df["open"] = df["open"].fillna(df["close"] * o2c)
    df["high"] = df["high"].fillna(df["close"] * h2c)
    df["low"] = df["low"].fillna(df["close"] * l2c)
    df["volume"] = df["volume"].fillna(df["volume"].median() if known_mask.sum() > 0 else 0)
    
    return df


def layer4_knn_fill(master: pd.DataFrame) -> pd.DataFrame:
    """
    Layer 4: EWMA + interpolation for all remaining gaps.
    """
    print("\n=== LAYER 4: Statistical Fill (all remaining) ===")
    
    # Find tickers with remaining NaN
    remaining = master[master["close"].isna()]["ticker"].unique()
    print(f"  Tickers with remaining NaN: {len(remaining)}")
    
    if len(remaining) == 0:
        print("  No remaining gaps!")
        return master
    
    for ticker in tqdm(remaining, desc="KNN fill"):
        mask = master["ticker"] == ticker
        idx = master.loc[mask].index
        sub = master.loc[idx].copy()
        sub = knn_impute_ticker(sub)
        master.loc[idx] = sub
    
    return master


def final_coverage_report(master: pd.DataFrame) -> pd.DataFrame:
    """Generate final coverage report."""
    print("\n=== FINAL COVERAGE ===")
    
    coverage = master.groupby("ticker").apply(
        lambda g: g[["open", "high", "low", "close"]].notna().all(axis=1).sum()
    )
    coverage = coverage.reset_index(name="complete_days")
    coverage["pct"] = (coverage["complete_days"] / TARGET_DATES * 100).round(1)
    
    print(f"  Tickers: {len(coverage)}")
    print(f"  Full coverage (6286): {(coverage['complete_days'] == TARGET_DATES).sum()}")
    print(f"  ≥99%: {(coverage['complete_days'] >= TARGET_DATES * 0.99).sum()}")
    print(f"  ≥95%: {(coverage['complete_days'] >= TARGET_DATES * 0.95).sum()}")
    print(f"  ≥90%: {(coverage['complete_days'] >= TARGET_DATES * 0.90).sum()}")
    print(f"  Min: {coverage['complete_days'].min()}")
    
    # Check for any remaining NaN
    nan_count = master[["open", "high", "low", "close"]].isna().sum().sum()
    print(f"\n  Remaining NaN cells: {nan_count}")
    
    return coverage


def main():
    global args
    args = parse_args()
    
    print("=" * 60)
    print("FINAL FILL PIPELINE")
    print("=" * 60)
    print(f"Target tickers: {args.top_n}")
    print(f"Target dates: {args.target_dates}")
    print(f"Workers: {args.workers}")
    print(f"Mode: {'DRY RUN' if args.dry_run else 'LIVE'}")
    
    # Load
    df, dates = load_data()
    
    # Select top N
    top_tickers, master = select_top_tickers(df, args.top_n, args.target_dates)
    
    # Layer 1: Already done in select (trim to 6286)
    layer1_trim_incomplete(master)
    
    # Layer 2: Linear interpolation
    master = layer2_linear_interpolation(master)
    
    # Layer 3: ARIMA trend projection
    master = layer3_arima_fill(master)
    
    # Layer 4: KNN/EWMA fill
    master = layer4_knn_fill(master)
    
    # Final report
    coverage = final_coverage_report(master)
    
    # Save
    if not args.dry_run:
        print(f"\nSaving...")
        master.to_csv(OUTPUT_FILE, index=False)
        print(f"  Saved: {OUTPUT_FILE} ({len(master):,} rows)")
        coverage.to_csv(COVERAGE_FILE, index=False)
        print(f"  Saved: {COVERAGE_FILE}")
    else:
        print("\nDRY RUN — no files saved.")
    
    print("\nDone.")


if __name__ == "__main__":
    main()

## data\yfin_engineer_features.py

In [ ]:
#!/usr/bin/env python3
"""
data/yfin_engineer_features.py

Build all market-derived feature files for the 2,500 ticker universe.
Memory-optimized: processes in chunks of 200 tickers.
No lookahead bias - all features strictly backward-looking.

Outputs:
  data/yFinance/processed/returns_panel_wide.csv    - 2500 tickers × 6285 days
  data/yFinance/processed/returns_long.csv          - ticker, date, log_return, simple_return
  data/yFinance/processed/liquidity_features.csv    - volume metrics
  data/yFinance/processed/features_temporal.csv     - 10 features for Temporal Encoder

Usage:
  python data/yfin_engineer_features.py --workers 6
"""

import os
import sys
import argparse
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────
INPUT_FILE   = Path("data/yFinance/processed/ohlcv_final.csv")
DATES_FILE   = Path("data/market_dates_ONLY_NYSE.csv")
OUT_DIR      = Path("data/yFinance/processed")
TICKERS_FILE = Path("data/yFinance/processed/common_tickers.csv")

RETURNS_WIDE = OUT_DIR / "returns_panel_wide.csv"
RETURNS_LONG = OUT_DIR / "returns_long.csv"
LIQUIDITY    = OUT_DIR / "liquidity_features.csv"
FEATURES     = OUT_DIR / "features_temporal.csv"

CHUNK_SIZE = 200      # tickers per chunk
MIN_VOL_WINDOW = 5     # minimum days before computing volatility

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--workers", type=int, default=4)
    return p.parse_args()


# ══════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING FUNCTIONS
# ══════════════════════════════════════════════════════════════

def rsi(close_series: pd.Series, window: int = 14) -> pd.Series:
    """Wilder's RSI — strictly backward-looking."""
    delta = close_series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def macd_hist(close_series: pd.Series) -> pd.Series:
    """MACD histogram: MACD - Signal line."""
    ema12 = close_series.ewm(span=12, adjust=False).mean()
    ema26 = close_series.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal = macd_line.ewm(span=9, adjust=False).mean()
    return macd_line - signal


def bollinger_position(close_series: pd.Series, window: int = 20) -> pd.Series:
    """Where is price within Bollinger Bands? 0 = lower, 1 = upper."""
    sma = close_series.rolling(window).mean()
    std = close_series.rolling(window).std()
    upper = sma + 2 * std
    lower = sma - 2 * std
    return (close_series - lower) / (upper - lower).replace(0, np.nan)


def price_position_21(close_series: pd.Series) -> pd.Series:
    """Price position within 21-day range. 0 = 21d low, 1 = 21d high."""
    h = close_series.rolling(21).max()
    l = close_series.rolling(21).min()
    return (close_series - l) / (h - l).replace(0, np.nan)


def process_ticker_block(ticker: str, df: pd.DataFrame, spy_returns: pd.Series) -> dict:
    """
    Process one ticker's full history. Returns dict of DataFrames
    for each output file type.
    """
    df = df.sort_values("date").copy()
    close = df["close"]
    
    # ── Returns ──────────────────────────────────────────
    # All strictly backward-looking: pct_change uses previous close
    df["log_return"]   = np.log(close / close.shift(1))
    df["simple_return"] = close.pct_change()
    
    # ── Volatility (annualized) ──────────────────────────
    df["vol_5d"]  = df["log_return"].rolling(5).std() * np.sqrt(252)
    df["vol_21d"] = df["log_return"].rolling(21).std() * np.sqrt(252)
    
    # ── Technical indicators ─────────────────────────────
    df["rsi_14"]    = rsi(close, 14)
    df["macd_hist"] = macd_hist(close)
    df["bb_pos"]    = bollinger_position(close, 20)
    df["price_pos"] = price_position_21(close)
    
    # ── Range ────────────────────────────────────────────
    df["hl_ratio"] = (df["high"] - df["low"]) / close.replace(0, np.nan)
    
    # ── Volume ───────────────────────────────────────────
    vol_mean_21 = df["volume"].rolling(21).mean()
    vol_std_21  = df["volume"].rolling(21).std().replace(0, np.nan)
    df["dollar_volume"]  = close * df["volume"]
    df["volume_zscore"]  = (df["volume"] - vol_mean_21) / vol_std_21
    df["volume_ratio"]   = df["volume"] / vol_mean_21.replace(0, np.nan)
    df["turnover_proxy"] = df["volume"] / df["volume"].rolling(252).mean().replace(0, np.nan)
    
    # ── SPY correlation ──────────────────────────────────
    if spy_returns is not None:
        
        # Align SPY returns to same index
        spy_aligned = pd.Series(spy_returns, index=df.index)
        df["spy_corr_63d"] = df["log_return"].rolling(63).corr(spy_aligned)
    else:
        df["spy_corr_63d"] = np.nan
    
    # ── Build output DataFrames ──────────────────────────
    
    # Returns long
    ret_long = df[["date", "ticker", "log_return", "simple_return"]].copy()
    
    # Liquidity
    liq = df[["date", "ticker", "dollar_volume", "volume_zscore", 
              "volume_ratio", "turnover_proxy"]].copy()
    
    # Temporal features (10 features for encoder)
    feat = df[["date", "ticker",
               "log_return", "vol_5d", "vol_21d", 
               "rsi_14", "macd_hist", "bb_pos",
               "volume_ratio", "hl_ratio", "price_pos",
               "spy_corr_63d"]].copy()
    
    return {
        "return": df[["date", "log_return"]].copy(),
        "ret_long": ret_long,
        "liquidity": liq,
        "features": feat,
    }


# ══════════════════════════════════════════════════════════════
#  MAIN PIPELINE
# ══════════════════════════════════════════════════════════════

def main():
    args = parse_args()
    
    print("=" * 60)
    print("FEATURE ENGINEERING PIPELINE")
    print("=" * 60)
    print(f"Input: {INPUT_FILE}")
    print(f"Chunk size: {CHUNK_SIZE} tickers")
    print()
    
    # ── Load SPY for correlation ─────────────────────────
    print("Loading SPY data for benchmark correlation...")
    spy_data = None
    spy_returns = None
    for chunk in pd.read_csv(INPUT_FILE, dtype={"ticker": str, "date": str}, chunksize=500000):
        spy_chunk = chunk[chunk["ticker"] == "SPY"]
        if len(spy_chunk) > 0:
            if spy_data is None:
                spy_data = spy_chunk
            else:
                spy_data = pd.concat([spy_data, spy_chunk])
    if spy_data is not None:
        spy_data = spy_data.sort_values("date")
        spy_data["log_return"] = np.log(spy_data["close"] / spy_data["close"].shift(1))
        spy_returns = spy_data["log_return"].values
        print(f"  SPY: {len(spy_data)} rows")
    else:
        print("  ⚠️  SPY not found — skipping correlation feature")
    
    # ── Get ticker list ──────────────────────────────────
    tickers = []
    for chunk in pd.read_csv(INPUT_FILE, dtype={"ticker": str}, 
                             usecols=["ticker"], chunksize=500000):
        tickers.extend(chunk["ticker"].unique())
    tickers = sorted(set(tickers))
    print(f"\nTotal tickers: {len(tickers)}")
    
    # ── Process in chunks ────────────────────────────────
    ticker_chunks = [tickers[i:i+CHUNK_SIZE] for i in range(0, len(tickers), CHUNK_SIZE)]
    print(f"Processing {len(ticker_chunks)} chunks of ≤{CHUNK_SIZE} tickers...\n")
    
    # Accumulators for wide returns
    returns_wide_parts = []
    first_ret_long = True
    first_liq = True
    first_feat = True
    
    for chunk_idx, ticker_chunk in enumerate(ticker_chunks):
        # Load only these tickers
        print(f"Chunk {chunk_idx+1}/{len(ticker_chunks)}: loading {len(ticker_chunk)} tickers...")
        
        frames = []
        for chunk in pd.read_csv(INPUT_FILE, dtype={"ticker": str, "date": str}, chunksize=1000000):
            sub = chunk[chunk["ticker"].isin(ticker_chunk)]
            if len(sub) > 0:
                frames.append(sub)
        
        if not frames:
            continue
        panel = pd.concat(frames, ignore_index=True)
        panel["date"] = pd.to_datetime(panel["date"])
        panel = panel.sort_values(["ticker", "date"])
        
        # Process each ticker
        ret_wide_dict = {}
        ret_long_list = []
        liq_list = []
        feat_list = []
        
        for ticker in tqdm(ticker_chunk, desc=f"  Chunk {chunk_idx+1}"):
            sub = panel[panel["ticker"] == ticker]
            if len(sub) < 10:
                continue
            
            result = process_ticker_block(ticker, sub, spy_returns)
            
            # Store return for wide matrix (skip first row = NaN)
            ret_series = result["return"].set_index("date")["log_return"]
            ret_series = ret_series.iloc[1:]  # drop first NaN
            if len(ret_series) > 0:
                ret_wide_dict[ticker] = ret_series
            
            ret_long_list.append(result["ret_long"])
            liq_list.append(result["liquidity"])
            feat_list.append(result["features"])
        
        # ── Write chunk outputs (append mode) ────────────
        
        # Returns wide: accumulate for final pivot
        if ret_wide_dict:
            returns_wide_parts.append(pd.DataFrame(ret_wide_dict))
        
        # Returns long
        if ret_long_list:
            df_long = pd.concat(ret_long_list, ignore_index=True)
            df_long = df_long[df_long["log_return"].notna()]
            if first_ret_long:
                df_long.to_csv(RETURNS_LONG, index=False, mode="w")
                first_ret_long = False
            else:
                df_long.to_csv(RETURNS_LONG, index=False, mode="a", header=False)
        
        # Liquidity
        if liq_list:
            df_liq = pd.concat(liq_list, ignore_index=True)
            if first_liq:
                df_liq.to_csv(LIQUIDITY, index=False, mode="w")
                first_liq = False
            else:
                df_liq.to_csv(LIQUIDITY, index=False, mode="a", header=False)
        
        # Temporal features
        if feat_list:
            df_feat = pd.concat(feat_list, ignore_index=True)
            if first_feat:
                df_feat.to_csv(FEATURES, index=False, mode="w")
                first_feat = False
            else:
                df_feat.to_csv(FEATURES, index=False, mode="a", header=False)
        
        del panel, frames, ret_long_list, liq_list, feat_list
    
    # ── Build wide returns matrix ────────────────────────
    print("\nBuilding returns_panel_wide.csv...")
    if returns_wide_parts:
        wide = pd.concat(returns_wide_parts, axis=1)
        wide = wide.sort_index()
        wide.to_csv(RETURNS_WIDE)
        print(f"  Shape: {wide.shape[0]} dates × {wide.shape[1]} tickers")
        print(f"  Saved: {RETURNS_WIDE}")
    
    # ── Summary ──────────────────────────────────────────
    print("\n" + "=" * 60)
    print("OUTPUT FILES")
    print("=" * 60)
    for name, path in [
        ("Returns Wide", RETURNS_WIDE),
        ("Returns Long", RETURNS_LONG),
        ("Liquidity Features", LIQUIDITY),
        ("Temporal Features (10 cols)", FEATURES),
    ]:
        if path.exists():
            size_mb = path.stat().st_size / 1024**2
            print(f"  ✅ {name}: {path} ({size_mb:.1f} MB)")
        else:
            print(f"  ❌ {name}: NOT CREATED")
    
    print("\nDone.")


if __name__ == "__main__":
    main()
